# ZVCMM125 — Nota fiscal eletronica — visao logistica — Análise Exploratória

**Tabela:** `dev_procurement.corp_curated.tbl_ds_log_zvcmm125`
**Transação SAP:** ZVCMM125
**Colunas:** 106 · **Clustering declarado:** `nr_nota_fiscal_eletronica`, `cd_empresa`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória entre elas.

## Alertas de partida _(extraídos dos comentários do `DESCRIBE`)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | **PK declarada de 5 campos, clustering de 2.** Os comentários marcam como PK: `nr_nota_fiscal_eletronica + cd_serie + cd_empresa + cd_fornecedor + cd_material`, mas o clustering só tem os dois primeiros. A seção 4 confirma qual identifica a linha. |
| 🔴 ALTA | **Nenhuma coluna de auditoria de carga** — sem `dateingest` ou `dh_carga` (seção 17). |
| 🟡 MEDIA | **30 colunas `double`** para valores fiscais — tolerância de 0,005 na conciliação. |
| 🟡 MEDIA | **4 unidades de medida diferentes** para a mesma linha: `cd_unidade_medida`, `_xml`, `_recebimento` e `_nfe`. Divergência entre elas é esperada, não erro (seção 18.4). |
| 🟡 MEDIA | **Forte semelhança com a ZGLMM425** (106 vs 93 colunas, mesmo clustering). Esta tem centro e vínculo com remessa/ordem; a outra tem dados de CT-e e SEFAZ. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas |
| 12.2 | Campos de hora |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| **18** | **Análises específicas da transação** |
| 19 | Amostra ampliada |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

In [0]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

In [0]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_log_zvcmm125 LIMIT 20;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                                        AS linhas,
       COUNT(DISTINCT CONCAT_WS('|', `nr_nota_fiscal_eletronica`, `cd_serie`, `cd_empresa`, `cd_fornecedor`, `cd_material`, `nr_item_nota_fiscal`)) AS chaves_completas,
       COUNT(DISTINCT `cd_centro`)                                 AS valores_de_cd_centro,
       COUNT(DISTINCT `nr_nota_fiscal_eletronica`)                                 AS nr_nota_fiscal_eletronica_distintos
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas)
que contenha os casos-limite identificados nas demais seções.

In [0]:
-- 3. DISTRIBUICAO POR cd_centro
SELECT COALESCE(NULLIF(trim(CAST(`cd_centro` AS STRING)), ''), '(vazio)') AS `cd_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY COALESCE(NULLIF(trim(CAST(`cd_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125),
g AS (
  SELECT 'nr_nota_fiscal_eletronica' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
UNION ALL
  SELECT 'nr_nota_fiscal_eletronica + cd_serie + cd_empresa + cd_fornecedor' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica`, `cd_serie`, `cd_empresa`, `cd_fornecedor` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
UNION ALL
  SELECT 'nr_nota_fiscal_eletronica + cd_serie + cd_empresa + cd_fornecedor + cd_material' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica`, `cd_serie`, `cd_empresa`, `cd_fornecedor`, `cd_material` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
UNION ALL
  SELECT 'nr_nota_fiscal_eletronica + cd_serie + cd_empresa + cd_fornecedor + cd_material + nr_item_nota_fiscal' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica`, `cd_serie`, `cd_empresa`, `cd_fornecedor`, `cd_material`, `nr_item_nota_fiscal` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `nr_nota_fiscal_eletronica + cd_empresa`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `nr_nota_fiscal_eletronica`, `cd_empresa`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY `nr_nota_fiscal_eletronica`, `cd_empresa`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (
  SELECT `nr_nota_fiscal_eletronica`, `cd_empresa` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `nr_nota_fiscal_eletronica`, `cd_empresa` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 c JOIN dup ON c.`nr_nota_fiscal_eletronica` <=> dup.`nr_nota_fiscal_eletronica` AND c.`cd_empresa` <=> dup.`cd_empresa`
),
agg AS (
  SELECT `nr_nota_fiscal_eletronica`, `cd_empresa`,
         COUNT(DISTINCT `cd_serie`) AS `cd_serie`,
         COUNT(DISTINCT `cd_local_negocios`) AS `cd_local_negocios`,
         COUNT(DISTINCT `cd_fornecedor`) AS `cd_fornecedor`,
         COUNT(DISTINCT `cd_chave_acesso_nfe`) AS `cd_chave_acesso_nfe`,
         COUNT(DISTINCT `nr_item_nota_fiscal`) AS `nr_item_nota_fiscal`,
         COUNT(DISTINCT `cd_material`) AS `cd_material`,
         COUNT(DISTINCT `cd_material_xml`) AS `cd_material_xml`,
         COUNT(DISTINCT `nr_documento_compras`) AS `nr_documento_compras`,
         COUNT(DISTINCT `nr_item_pedido`) AS `nr_item_pedido`,
         COUNT(DISTINCT `nr_documento_faturamento`) AS `nr_documento_faturamento`,
         COUNT(DISTINCT `nr_exercicio`) AS `nr_exercicio`,
         COUNT(DISTINCT `nr_documento_material`) AS `nr_documento_material`,
         COUNT(DISTINCT `nr_remessa`) AS `nr_remessa`,
         COUNT(DISTINCT `cd_chave_acesso_dacte`) AS `cd_chave_acesso_dacte`,
         COUNT(DISTINCT `cd_cnpj_transportador`) AS `cd_cnpj_transportador`,
         COUNT(DISTINCT `cd_cnpj`) AS `cd_cnpj`,
         COUNT(DISTINCT `cd_codigo_barras`) AS `cd_codigo_barras`,
         COUNT(DISTINCT `qt_contagem_cte`) AS `qt_contagem_cte`,
         COUNT(DISTINCT `nr_transporte`) AS `nr_transporte`,
         COUNT(DISTINCT `cd_centro`) AS `cd_centro`,
         COUNT(DISTINCT `nm_centro`) AS `nm_centro`,
         COUNT(DISTINCT `dt_documento`) AS `dt_documento`,
         COUNT(DISTINCT `dt_recebimento`) AS `dt_recebimento`,
         COUNT(DISTINCT `dt_recebimento_item`) AS `dt_recebimento_item`,
         COUNT(DISTINCT `dt_justificativa`) AS `dt_justificativa`,
         COUNT(DISTINCT `dt_data`) AS `dt_data`,
         COUNT(DISTINCT `dt_lancamento`) AS `dt_lancamento`,
         COUNT(DISTINCT `dt_vencimento_duplicata`) AS `dt_vencimento_duplicata`,
         COUNT(DISTINCT `hr_hora`) AS `hr_hora`,
         COUNT(DISTINCT `cd_ncm`) AS `cd_ncm`,
         COUNT(DISTINCT `cd_cfop`) AS `cd_cfop`,
         COUNT(DISTINCT `cd_cst_icms`) AS `cd_cst_icms`,
         COUNT(DISTINCT `cd_iva`) AS `cd_iva`,
         COUNT(DISTINCT `cd_origem_material`) AS `cd_origem_material`,
         COUNT(DISTINCT `cd_procedencia_material`) AS `cd_procedencia_material`,
         COUNT(DISTINCT `tx_descricao_produto`) AS `tx_descricao_produto`,
         COUNT(DISTINCT `tx_descricao_material`) AS `tx_descricao_material`,
         COUNT(DISTINCT `qt_quantidade`) AS `qt_quantidade`,
         COUNT(DISTINCT `qt_quantidade_xml`) AS `qt_quantidade_xml`,
         COUNT(DISTINCT `qt_quantidade_recebida`) AS `qt_quantidade_recebida`,
         COUNT(DISTINCT `cd_unidade_medida`) AS `cd_unidade_medida`,
         COUNT(DISTINCT `cd_unidade_medida_xml`) AS `cd_unidade_medida_xml`,
         COUNT(DISTINCT `cd_unidade_medida_recebimento`) AS `cd_unidade_medida_recebimento`,
         COUNT(DISTINCT `cd_unidade_medida_nfe`) AS `cd_unidade_medida_nfe`,
         COUNT(DISTINCT `vl_preco_unitario_bruto`) AS `vl_preco_unitario_bruto`,
         COUNT(DISTINCT `vl_preco_unitario_liquido`) AS `vl_preco_unitario_liquido`,
         COUNT(DISTINCT `vl_unitario`) AS `vl_unitario`,
         COUNT(DISTINCT `vl_bruto`) AS `vl_bruto`,
         COUNT(DISTINCT `vl_liquido`) AS `vl_liquido`,
         COUNT(DISTINCT `vl_total`) AS `vl_total`,
         COUNT(DISTINCT `pc_aliquota_icms`) AS `pc_aliquota_icms`,
         COUNT(DISTINCT `vl_base_icms`) AS `vl_base_icms`,
         COUNT(DISTINCT `vl_icms`) AS `vl_icms`,
         COUNT(DISTINCT `pc_aliquota_st`) AS `pc_aliquota_st`,
         COUNT(DISTINCT `vl_base_st`) AS `vl_base_st`,
         COUNT(DISTINCT `vl_st`) AS `vl_st`,
         COUNT(DISTINCT `pc_aliquota_ipi`) AS `pc_aliquota_ipi`,
         COUNT(DISTINCT `vl_base_ipi`) AS `vl_base_ipi`,
         COUNT(DISTINCT `vl_ipi`) AS `vl_ipi`,
         COUNT(DISTINCT `pc_aliquota_pis`) AS `pc_aliquota_pis`,
         COUNT(DISTINCT `vl_base_pis`) AS `vl_base_pis`,
         COUNT(DISTINCT `vl_pis`) AS `vl_pis`,
         COUNT(DISTINCT `pc_aliquota_cofins`) AS `pc_aliquota_cofins`,
         COUNT(DISTINCT `vl_base_cofins`) AS `vl_base_cofins`,
         COUNT(DISTINCT `vl_cofins`) AS `vl_cofins`,
         COUNT(DISTINCT `vl_base_calculo_icms_item`) AS `vl_base_calculo_icms_item`,
         COUNT(DISTINCT `vl_total_bruto`) AS `vl_total_bruto`,
         COUNT(DISTINCT `vl_total_liquido`) AS `vl_total_liquido`,
         COUNT(DISTINCT `vl_total_icms`) AS `vl_total_icms`,
         COUNT(DISTINCT `vl_total_st`) AS `vl_total_st`,
         COUNT(DISTINCT `vl_total_ipi`) AS `vl_total_ipi`,
         COUNT(DISTINCT `vl_total_produto`) AS `vl_total_produto`,
         COUNT(DISTINCT `vl_despesas_acessorias`) AS `vl_despesas_acessorias`,
         COUNT(DISTINCT `vl_frete`) AS `vl_frete`,
         COUNT(DISTINCT `vl_icms_nao_destacado`) AS `vl_icms_nao_destacado`,
         COUNT(DISTINCT `vl_icms_st_nao_destacado`) AS `vl_icms_st_nao_destacado`,
         COUNT(DISTINCT `vl_base_calculo_icms`) AS `vl_base_calculo_icms`,
         COUNT(DISTINCT `vl_base_calculo_st`) AS `vl_base_calculo_st`,
         COUNT(DISTINCT `qt_peso_total_nota_fiscal`) AS `qt_peso_total_nota_fiscal`,
         COUNT(DISTINCT `qt_peso_liquido`) AS `qt_peso_liquido`,
         COUNT(DISTINCT `cd_unidade_peso`) AS `cd_unidade_peso`,
         COUNT(DISTINCT `qt_volumes`) AS `qt_volumes`,
         COUNT(DISTINCT `cd_incoterms`) AS `cd_incoterms`,
         COUNT(DISTINCT `cd_modalidade_frete`) AS `cd_modalidade_frete`,
         COUNT(DISTINCT `nm_transportador`) AS `nm_transportador`,
         COUNT(DISTINCT `cd_frente`) AS `cd_frente`,
         COUNT(DISTINCT `tx_status`) AS `tx_status`,
         COUNT(DISTINCT `st_simulacao`) AS `st_simulacao`,
         COUNT(DISTINCT `tx_log`) AS `tx_log`,
         COUNT(DISTINCT `tx_log_simulacao`) AS `tx_log_simulacao`,
         COUNT(DISTINCT `tx_justificativa`) AS `tx_justificativa`,
         COUNT(DISTINCT `nm_usuario`) AS `nm_usuario`,
         COUNT(DISTINCT `cd_area_responsavel`) AS `cd_area_responsavel`,
         COUNT(DISTINCT `tp_documento_compras`) AS `tp_documento_compras`,
         COUNT(DISTINCT `tp_classificacao_contabil`) AS `tp_classificacao_contabil`,
         COUNT(DISTINCT `cd_grupo_mercadorias`) AS `cd_grupo_mercadorias`,
         COUNT(DISTINCT `dt_pedido_compra`) AS `dt_pedido_compra`,
         COUNT(DISTINCT `vl_preco_liquido_pedido`) AS `vl_preco_liquido_pedido`,
         COUNT(DISTINCT `nr_remessa_entrega`) AS `nr_remessa_entrega`,
         COUNT(DISTINCT `dt_criacao_remessa`) AS `dt_criacao_remessa`,
         COUNT(DISTINCT `hr_criacao_remessa`) AS `hr_criacao_remessa`,
         COUNT(DISTINCT `nr_ordem`) AS `nr_ordem`,
         COUNT(DISTINCT `cd_revisao_manutencao`) AS `cd_revisao_manutencao`,
         COUNT(DISTINCT `tp_mrp`) AS `tp_mrp`
  FROM d GROUP BY `nr_nota_fiscal_eletronica`, `cd_empresa`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(104,
    'cd_serie', MAX(`cd_serie`),
    'cd_local_negocios', MAX(`cd_local_negocios`),
    'cd_fornecedor', MAX(`cd_fornecedor`),
    'cd_chave_acesso_nfe', MAX(`cd_chave_acesso_nfe`),
    'nr_item_nota_fiscal', MAX(`nr_item_nota_fiscal`),
    'cd_material', MAX(`cd_material`),
    'cd_material_xml', MAX(`cd_material_xml`),
    'nr_documento_compras', MAX(`nr_documento_compras`),
    'nr_item_pedido', MAX(`nr_item_pedido`),
    'nr_documento_faturamento', MAX(`nr_documento_faturamento`),
    'nr_exercicio', MAX(`nr_exercicio`),
    'nr_documento_material', MAX(`nr_documento_material`),
    'nr_remessa', MAX(`nr_remessa`),
    'cd_chave_acesso_dacte', MAX(`cd_chave_acesso_dacte`),
    'cd_cnpj_transportador', MAX(`cd_cnpj_transportador`),
    'cd_cnpj', MAX(`cd_cnpj`),
    'cd_codigo_barras', MAX(`cd_codigo_barras`),
    'qt_contagem_cte', MAX(`qt_contagem_cte`),
    'nr_transporte', MAX(`nr_transporte`),
    'cd_centro', MAX(`cd_centro`),
    'nm_centro', MAX(`nm_centro`),
    'dt_documento', MAX(`dt_documento`),
    'dt_recebimento', MAX(`dt_recebimento`),
    'dt_recebimento_item', MAX(`dt_recebimento_item`),
    'dt_justificativa', MAX(`dt_justificativa`),
    'dt_data', MAX(`dt_data`),
    'dt_lancamento', MAX(`dt_lancamento`),
    'dt_vencimento_duplicata', MAX(`dt_vencimento_duplicata`),
    'hr_hora', MAX(`hr_hora`),
    'cd_ncm', MAX(`cd_ncm`),
    'cd_cfop', MAX(`cd_cfop`),
    'cd_cst_icms', MAX(`cd_cst_icms`),
    'cd_iva', MAX(`cd_iva`),
    'cd_origem_material', MAX(`cd_origem_material`),
    'cd_procedencia_material', MAX(`cd_procedencia_material`),
    'tx_descricao_produto', MAX(`tx_descricao_produto`),
    'tx_descricao_material', MAX(`tx_descricao_material`),
    'qt_quantidade', MAX(`qt_quantidade`),
    'qt_quantidade_xml', MAX(`qt_quantidade_xml`),
    'qt_quantidade_recebida', MAX(`qt_quantidade_recebida`),
    'cd_unidade_medida', MAX(`cd_unidade_medida`),
    'cd_unidade_medida_xml', MAX(`cd_unidade_medida_xml`),
    'cd_unidade_medida_recebimento', MAX(`cd_unidade_medida_recebimento`),
    'cd_unidade_medida_nfe', MAX(`cd_unidade_medida_nfe`),
    'vl_preco_unitario_bruto', MAX(`vl_preco_unitario_bruto`),
    'vl_preco_unitario_liquido', MAX(`vl_preco_unitario_liquido`),
    'vl_unitario', MAX(`vl_unitario`),
    'vl_bruto', MAX(`vl_bruto`),
    'vl_liquido', MAX(`vl_liquido`),
    'vl_total', MAX(`vl_total`),
    'pc_aliquota_icms', MAX(`pc_aliquota_icms`),
    'vl_base_icms', MAX(`vl_base_icms`),
    'vl_icms', MAX(`vl_icms`),
    'pc_aliquota_st', MAX(`pc_aliquota_st`),
    'vl_base_st', MAX(`vl_base_st`),
    'vl_st', MAX(`vl_st`),
    'pc_aliquota_ipi', MAX(`pc_aliquota_ipi`),
    'vl_base_ipi', MAX(`vl_base_ipi`),
    'vl_ipi', MAX(`vl_ipi`),
    'pc_aliquota_pis', MAX(`pc_aliquota_pis`),
    'vl_base_pis', MAX(`vl_base_pis`),
    'vl_pis', MAX(`vl_pis`),
    'pc_aliquota_cofins', MAX(`pc_aliquota_cofins`),
    'vl_base_cofins', MAX(`vl_base_cofins`),
    'vl_cofins', MAX(`vl_cofins`),
    'vl_base_calculo_icms_item', MAX(`vl_base_calculo_icms_item`),
    'vl_total_bruto', MAX(`vl_total_bruto`),
    'vl_total_liquido', MAX(`vl_total_liquido`),
    'vl_total_icms', MAX(`vl_total_icms`),
    'vl_total_st', MAX(`vl_total_st`),
    'vl_total_ipi', MAX(`vl_total_ipi`),
    'vl_total_produto', MAX(`vl_total_produto`),
    'vl_despesas_acessorias', MAX(`vl_despesas_acessorias`),
    'vl_frete', MAX(`vl_frete`),
    'vl_icms_nao_destacado', MAX(`vl_icms_nao_destacado`),
    'vl_icms_st_nao_destacado', MAX(`vl_icms_st_nao_destacado`),
    'vl_base_calculo_icms', MAX(`vl_base_calculo_icms`),
    'vl_base_calculo_st', MAX(`vl_base_calculo_st`),
    'qt_peso_total_nota_fiscal', MAX(`qt_peso_total_nota_fiscal`),
    'qt_peso_liquido', MAX(`qt_peso_liquido`),
    'cd_unidade_peso', MAX(`cd_unidade_peso`),
    'qt_volumes', MAX(`qt_volumes`),
    'cd_incoterms', MAX(`cd_incoterms`),
    'cd_modalidade_frete', MAX(`cd_modalidade_frete`),
    'nm_transportador', MAX(`nm_transportador`),
    'cd_frente', MAX(`cd_frente`),
    'tx_status', MAX(`tx_status`),
    'st_simulacao', MAX(`st_simulacao`),
    'tx_log', MAX(`tx_log`),
    'tx_log_simulacao', MAX(`tx_log_simulacao`),
    'tx_justificativa', MAX(`tx_justificativa`),
    'nm_usuario', MAX(`nm_usuario`),
    'cd_area_responsavel', MAX(`cd_area_responsavel`),
    'tp_documento_compras', MAX(`tp_documento_compras`),
    'tp_classificacao_contabil', MAX(`tp_classificacao_contabil`),
    'cd_grupo_mercadorias', MAX(`cd_grupo_mercadorias`),
    'dt_pedido_compra', MAX(`dt_pedido_compra`),
    'vl_preco_liquido_pedido', MAX(`vl_preco_liquido_pedido`),
    'nr_remessa_entrega', MAX(`nr_remessa_entrega`),
    'dt_criacao_remessa', MAX(`dt_criacao_remessa`),
    'hr_criacao_remessa', MAX(`hr_criacao_remessa`),
    'nr_ordem', MAX(`nr_ordem`),
    'cd_revisao_manutencao', MAX(`cd_revisao_manutencao`),
    'tp_mrp', MAX(`tp_mrp`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Em validação anterior, esta análise revelou 7 colunas 100% nulas no Datalake —
uma delas preenchida em 97,9% dos registros do SAP. Este erro **não aparece** em amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

> **59 colunas não têm comentário** no `DESCRIBE` desta tabela. Coluna sem descrição costuma ser a assinatura de campo nunca carregado — confira o `veredito` delas com atenção.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, 'zeros_ou_false' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125),
perf AS (
  SELECT stack(106,
    'nr_nota_fiscal_eletronica', 'string', COUNT_IF(`nr_nota_fiscal_eletronica` IS NULL), COUNT_IF(`nr_nota_fiscal_eletronica` IS NOT NULL AND lower(trim(`nr_nota_fiscal_eletronica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_nota_fiscal_eletronica`) RLIKE '^0+([.,]0+)?$'),
    'cd_serie', 'string', COUNT_IF(`cd_serie` IS NULL), COUNT_IF(`cd_serie` IS NOT NULL AND lower(trim(`cd_serie`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_serie`) RLIKE '^0+([.,]0+)?$'),
    'cd_empresa', 'string', COUNT_IF(`cd_empresa` IS NULL), COUNT_IF(`cd_empresa` IS NOT NULL AND lower(trim(`cd_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cd_local_negocios', 'string', COUNT_IF(`cd_local_negocios` IS NULL), COUNT_IF(`cd_local_negocios` IS NOT NULL AND lower(trim(`cd_local_negocios`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_local_negocios`) RLIKE '^0+([.,]0+)?$'),
    'cd_fornecedor', 'string', COUNT_IF(`cd_fornecedor` IS NULL), COUNT_IF(`cd_fornecedor` IS NOT NULL AND lower(trim(`cd_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cd_chave_acesso_nfe', 'string', COUNT_IF(`cd_chave_acesso_nfe` IS NULL), COUNT_IF(`cd_chave_acesso_nfe` IS NOT NULL AND lower(trim(`cd_chave_acesso_nfe`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_chave_acesso_nfe`) RLIKE '^0+([.,]0+)?$'),
    'nr_item_nota_fiscal', 'string', COUNT_IF(`nr_item_nota_fiscal` IS NULL), COUNT_IF(`nr_item_nota_fiscal` IS NOT NULL AND lower(trim(`nr_item_nota_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_item_nota_fiscal`) RLIKE '^0+([.,]0+)?$'),
    'cd_material', 'string', COUNT_IF(`cd_material` IS NULL), COUNT_IF(`cd_material` IS NOT NULL AND lower(trim(`cd_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_material`) RLIKE '^0+([.,]0+)?$'),
    'cd_material_xml', 'string', COUNT_IF(`cd_material_xml` IS NULL), COUNT_IF(`cd_material_xml` IS NOT NULL AND lower(trim(`cd_material_xml`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_material_xml`) RLIKE '^0+([.,]0+)?$'),
    'nr_documento_compras', 'string', COUNT_IF(`nr_documento_compras` IS NULL), COUNT_IF(`nr_documento_compras` IS NOT NULL AND lower(trim(`nr_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_documento_compras`) RLIKE '^0+([.,]0+)?$'),
    'nr_item_pedido', 'string', COUNT_IF(`nr_item_pedido` IS NULL), COUNT_IF(`nr_item_pedido` IS NOT NULL AND lower(trim(`nr_item_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_item_pedido`) RLIKE '^0+([.,]0+)?$'),
    'nr_documento_faturamento', 'string', COUNT_IF(`nr_documento_faturamento` IS NULL), COUNT_IF(`nr_documento_faturamento` IS NOT NULL AND lower(trim(`nr_documento_faturamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_documento_faturamento`) RLIKE '^0+([.,]0+)?$'),
    'nr_exercicio', 'string', COUNT_IF(`nr_exercicio` IS NULL), COUNT_IF(`nr_exercicio` IS NOT NULL AND lower(trim(`nr_exercicio`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_exercicio`) RLIKE '^0+([.,]0+)?$'),
    'nr_documento_material', 'string', COUNT_IF(`nr_documento_material` IS NULL), COUNT_IF(`nr_documento_material` IS NOT NULL AND lower(trim(`nr_documento_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_documento_material`) RLIKE '^0+([.,]0+)?$'),
    'nr_remessa', 'string', COUNT_IF(`nr_remessa` IS NULL), COUNT_IF(`nr_remessa` IS NOT NULL AND lower(trim(`nr_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_remessa`) RLIKE '^0+([.,]0+)?$'),
    'cd_chave_acesso_dacte', 'string', COUNT_IF(`cd_chave_acesso_dacte` IS NULL), COUNT_IF(`cd_chave_acesso_dacte` IS NOT NULL AND lower(trim(`cd_chave_acesso_dacte`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_chave_acesso_dacte`) RLIKE '^0+([.,]0+)?$'),
    'cd_cnpj_transportador', 'string', COUNT_IF(`cd_cnpj_transportador` IS NULL), COUNT_IF(`cd_cnpj_transportador` IS NOT NULL AND lower(trim(`cd_cnpj_transportador`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_cnpj_transportador`) RLIKE '^0+([.,]0+)?$'),
    'cd_cnpj', 'string', COUNT_IF(`cd_cnpj` IS NULL), COUNT_IF(`cd_cnpj` IS NOT NULL AND lower(trim(`cd_cnpj`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_cnpj`) RLIKE '^0+([.,]0+)?$'),
    'cd_codigo_barras', 'string', COUNT_IF(`cd_codigo_barras` IS NULL), COUNT_IF(`cd_codigo_barras` IS NOT NULL AND lower(trim(`cd_codigo_barras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_codigo_barras`) RLIKE '^0+([.,]0+)?$'),
    'qt_contagem_cte', 'int', COUNT_IF(`qt_contagem_cte` IS NULL), 0L, COUNT_IF(`qt_contagem_cte` = 0),
    'nr_transporte', 'string', COUNT_IF(`nr_transporte` IS NULL), COUNT_IF(`nr_transporte` IS NOT NULL AND lower(trim(`nr_transporte`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_transporte`) RLIKE '^0+([.,]0+)?$'),
    'cd_centro', 'string', COUNT_IF(`cd_centro` IS NULL), COUNT_IF(`cd_centro` IS NOT NULL AND lower(trim(`cd_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_centro`) RLIKE '^0+([.,]0+)?$'),
    'nm_centro', 'string', COUNT_IF(`nm_centro` IS NULL), COUNT_IF(`nm_centro` IS NOT NULL AND lower(trim(`nm_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_centro`) RLIKE '^0+([.,]0+)?$'),
    'dt_documento', 'date', COUNT_IF(`dt_documento` IS NULL), 0L, 0L,
    'dt_recebimento', 'date', COUNT_IF(`dt_recebimento` IS NULL), 0L, 0L,
    'dt_recebimento_item', 'date', COUNT_IF(`dt_recebimento_item` IS NULL), 0L, 0L,
    'dt_justificativa', 'date', COUNT_IF(`dt_justificativa` IS NULL), 0L, 0L,
    'dt_data', 'date', COUNT_IF(`dt_data` IS NULL), 0L, 0L,
    'dt_lancamento', 'date', COUNT_IF(`dt_lancamento` IS NULL), 0L, 0L,
    'dt_vencimento_duplicata', 'date', COUNT_IF(`dt_vencimento_duplicata` IS NULL), 0L, 0L,
    'hr_hora', 'string', COUNT_IF(`hr_hora` IS NULL), COUNT_IF(`hr_hora` IS NOT NULL AND lower(trim(`hr_hora`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`hr_hora`) RLIKE '^0+([.,]0+)?$'),
    'cd_ncm', 'string', COUNT_IF(`cd_ncm` IS NULL), COUNT_IF(`cd_ncm` IS NOT NULL AND lower(trim(`cd_ncm`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_ncm`) RLIKE '^0+([.,]0+)?$'),
    'cd_cfop', 'string', COUNT_IF(`cd_cfop` IS NULL), COUNT_IF(`cd_cfop` IS NOT NULL AND lower(trim(`cd_cfop`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_cfop`) RLIKE '^0+([.,]0+)?$'),
    'cd_cst_icms', 'string', COUNT_IF(`cd_cst_icms` IS NULL), COUNT_IF(`cd_cst_icms` IS NOT NULL AND lower(trim(`cd_cst_icms`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_cst_icms`) RLIKE '^0+([.,]0+)?$'),
    'cd_iva', 'string', COUNT_IF(`cd_iva` IS NULL), COUNT_IF(`cd_iva` IS NOT NULL AND lower(trim(`cd_iva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_iva`) RLIKE '^0+([.,]0+)?$'),
    'cd_origem_material', 'string', COUNT_IF(`cd_origem_material` IS NULL), COUNT_IF(`cd_origem_material` IS NOT NULL AND lower(trim(`cd_origem_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_origem_material`) RLIKE '^0+([.,]0+)?$'),
    'cd_procedencia_material', 'string', COUNT_IF(`cd_procedencia_material` IS NULL), COUNT_IF(`cd_procedencia_material` IS NOT NULL AND lower(trim(`cd_procedencia_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_procedencia_material`) RLIKE '^0+([.,]0+)?$'),
    'tx_descricao_produto', 'string', COUNT_IF(`tx_descricao_produto` IS NULL), COUNT_IF(`tx_descricao_produto` IS NOT NULL AND lower(trim(`tx_descricao_produto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tx_descricao_produto`) RLIKE '^0+([.,]0+)?$'),
    'tx_descricao_material', 'string', COUNT_IF(`tx_descricao_material` IS NULL), COUNT_IF(`tx_descricao_material` IS NOT NULL AND lower(trim(`tx_descricao_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tx_descricao_material`) RLIKE '^0+([.,]0+)?$'),
    'qt_quantidade', 'decimal(13,3)', COUNT_IF(`qt_quantidade` IS NULL), 0L, COUNT_IF(`qt_quantidade` = 0),
    'qt_quantidade_xml', 'decimal(13,3)', COUNT_IF(`qt_quantidade_xml` IS NULL), 0L, COUNT_IF(`qt_quantidade_xml` = 0),
    'qt_quantidade_recebida', 'decimal(13,3)', COUNT_IF(`qt_quantidade_recebida` IS NULL), 0L, COUNT_IF(`qt_quantidade_recebida` = 0),
    'cd_unidade_medida', 'string', COUNT_IF(`cd_unidade_medida` IS NULL), COUNT_IF(`cd_unidade_medida` IS NOT NULL AND lower(trim(`cd_unidade_medida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_unidade_medida`) RLIKE '^0+([.,]0+)?$'),
    'cd_unidade_medida_xml', 'string', COUNT_IF(`cd_unidade_medida_xml` IS NULL), COUNT_IF(`cd_unidade_medida_xml` IS NOT NULL AND lower(trim(`cd_unidade_medida_xml`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_unidade_medida_xml`) RLIKE '^0+([.,]0+)?$'),
    'cd_unidade_medida_recebimento', 'string', COUNT_IF(`cd_unidade_medida_recebimento` IS NULL), COUNT_IF(`cd_unidade_medida_recebimento` IS NOT NULL AND lower(trim(`cd_unidade_medida_recebimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_unidade_medida_recebimento`) RLIKE '^0+([.,]0+)?$'),
    'cd_unidade_medida_nfe', 'string', COUNT_IF(`cd_unidade_medida_nfe` IS NULL), COUNT_IF(`cd_unidade_medida_nfe` IS NOT NULL AND lower(trim(`cd_unidade_medida_nfe`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_unidade_medida_nfe`) RLIKE '^0+([.,]0+)?$'),
    'vl_preco_unitario_bruto', 'double', COUNT_IF(`vl_preco_unitario_bruto` IS NULL), 0L, COUNT_IF(`vl_preco_unitario_bruto` = 0),
    'vl_preco_unitario_liquido', 'double', COUNT_IF(`vl_preco_unitario_liquido` IS NULL), 0L, COUNT_IF(`vl_preco_unitario_liquido` = 0),
    'vl_unitario', 'double', COUNT_IF(`vl_unitario` IS NULL), 0L, COUNT_IF(`vl_unitario` = 0),
    'vl_bruto', 'double', COUNT_IF(`vl_bruto` IS NULL), 0L, COUNT_IF(`vl_bruto` = 0),
    'vl_liquido', 'double', COUNT_IF(`vl_liquido` IS NULL), 0L, COUNT_IF(`vl_liquido` = 0),
    'vl_total', 'double', COUNT_IF(`vl_total` IS NULL), 0L, COUNT_IF(`vl_total` = 0),
    'pc_aliquota_icms', 'decimal(7,2)', COUNT_IF(`pc_aliquota_icms` IS NULL), 0L, COUNT_IF(`pc_aliquota_icms` = 0),
    'vl_base_icms', 'double', COUNT_IF(`vl_base_icms` IS NULL), 0L, COUNT_IF(`vl_base_icms` = 0),
    'vl_icms', 'double', COUNT_IF(`vl_icms` IS NULL), 0L, COUNT_IF(`vl_icms` = 0),
    'pc_aliquota_st', 'decimal(7,2)', COUNT_IF(`pc_aliquota_st` IS NULL), 0L, COUNT_IF(`pc_aliquota_st` = 0),
    'vl_base_st', 'double', COUNT_IF(`vl_base_st` IS NULL), 0L, COUNT_IF(`vl_base_st` = 0),
    'vl_st', 'double', COUNT_IF(`vl_st` IS NULL), 0L, COUNT_IF(`vl_st` = 0),
    'pc_aliquota_ipi', 'decimal(7,2)', COUNT_IF(`pc_aliquota_ipi` IS NULL), 0L, COUNT_IF(`pc_aliquota_ipi` = 0),
    'vl_base_ipi', 'double', COUNT_IF(`vl_base_ipi` IS NULL), 0L, COUNT_IF(`vl_base_ipi` = 0),
    'vl_ipi', 'double', COUNT_IF(`vl_ipi` IS NULL), 0L, COUNT_IF(`vl_ipi` = 0),
    'pc_aliquota_pis', 'decimal(7,2)', COUNT_IF(`pc_aliquota_pis` IS NULL), 0L, COUNT_IF(`pc_aliquota_pis` = 0),
    'vl_base_pis', 'double', COUNT_IF(`vl_base_pis` IS NULL), 0L, COUNT_IF(`vl_base_pis` = 0),
    'vl_pis', 'double', COUNT_IF(`vl_pis` IS NULL), 0L, COUNT_IF(`vl_pis` = 0),
    'pc_aliquota_cofins', 'decimal(7,2)', COUNT_IF(`pc_aliquota_cofins` IS NULL), 0L, COUNT_IF(`pc_aliquota_cofins` = 0),
    'vl_base_cofins', 'double', COUNT_IF(`vl_base_cofins` IS NULL), 0L, COUNT_IF(`vl_base_cofins` = 0),
    'vl_cofins', 'double', COUNT_IF(`vl_cofins` IS NULL), 0L, COUNT_IF(`vl_cofins` = 0),
    'vl_base_calculo_icms_item', 'double', COUNT_IF(`vl_base_calculo_icms_item` IS NULL), 0L, COUNT_IF(`vl_base_calculo_icms_item` = 0),
    'vl_total_bruto', 'double', COUNT_IF(`vl_total_bruto` IS NULL), 0L, COUNT_IF(`vl_total_bruto` = 0),
    'vl_total_liquido', 'double', COUNT_IF(`vl_total_liquido` IS NULL), 0L, COUNT_IF(`vl_total_liquido` = 0),
    'vl_total_icms', 'double', COUNT_IF(`vl_total_icms` IS NULL), 0L, COUNT_IF(`vl_total_icms` = 0),
    'vl_total_st', 'double', COUNT_IF(`vl_total_st` IS NULL), 0L, COUNT_IF(`vl_total_st` = 0),
    'vl_total_ipi', 'double', COUNT_IF(`vl_total_ipi` IS NULL), 0L, COUNT_IF(`vl_total_ipi` = 0),
    'vl_total_produto', 'double', COUNT_IF(`vl_total_produto` IS NULL), 0L, COUNT_IF(`vl_total_produto` = 0),
    'vl_despesas_acessorias', 'double', COUNT_IF(`vl_despesas_acessorias` IS NULL), 0L, COUNT_IF(`vl_despesas_acessorias` = 0),
    'vl_frete', 'double', COUNT_IF(`vl_frete` IS NULL), 0L, COUNT_IF(`vl_frete` = 0),
    'vl_icms_nao_destacado', 'double', COUNT_IF(`vl_icms_nao_destacado` IS NULL), 0L, COUNT_IF(`vl_icms_nao_destacado` = 0),
    'vl_icms_st_nao_destacado', 'double', COUNT_IF(`vl_icms_st_nao_destacado` IS NULL), 0L, COUNT_IF(`vl_icms_st_nao_destacado` = 0),
    'vl_base_calculo_icms', 'double', COUNT_IF(`vl_base_calculo_icms` IS NULL), 0L, COUNT_IF(`vl_base_calculo_icms` = 0),
    'vl_base_calculo_st', 'double', COUNT_IF(`vl_base_calculo_st` IS NULL), 0L, COUNT_IF(`vl_base_calculo_st` = 0),
    'qt_peso_total_nota_fiscal', 'decimal(15,3)', COUNT_IF(`qt_peso_total_nota_fiscal` IS NULL), 0L, COUNT_IF(`qt_peso_total_nota_fiscal` = 0),
    'qt_peso_liquido', 'decimal(15,3)', COUNT_IF(`qt_peso_liquido` IS NULL), 0L, COUNT_IF(`qt_peso_liquido` = 0),
    'cd_unidade_peso', 'string', COUNT_IF(`cd_unidade_peso` IS NULL), COUNT_IF(`cd_unidade_peso` IS NOT NULL AND lower(trim(`cd_unidade_peso`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_unidade_peso`) RLIKE '^0+([.,]0+)?$'),
    'qt_volumes', 'string', COUNT_IF(`qt_volumes` IS NULL), COUNT_IF(`qt_volumes` IS NOT NULL AND lower(trim(`qt_volumes`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`qt_volumes`) RLIKE '^0+([.,]0+)?$'),
    'cd_incoterms', 'string', COUNT_IF(`cd_incoterms` IS NULL), COUNT_IF(`cd_incoterms` IS NOT NULL AND lower(trim(`cd_incoterms`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_incoterms`) RLIKE '^0+([.,]0+)?$'),
    'cd_modalidade_frete', 'string', COUNT_IF(`cd_modalidade_frete` IS NULL), COUNT_IF(`cd_modalidade_frete` IS NOT NULL AND lower(trim(`cd_modalidade_frete`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_modalidade_frete`) RLIKE '^0+([.,]0+)?$'),
    'nm_transportador', 'string', COUNT_IF(`nm_transportador` IS NULL), COUNT_IF(`nm_transportador` IS NOT NULL AND lower(trim(`nm_transportador`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_transportador`) RLIKE '^0+([.,]0+)?$'),
    'cd_frente', 'string', COUNT_IF(`cd_frente` IS NULL), COUNT_IF(`cd_frente` IS NOT NULL AND lower(trim(`cd_frente`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_frente`) RLIKE '^0+([.,]0+)?$'),
    'tx_status', 'string', COUNT_IF(`tx_status` IS NULL), COUNT_IF(`tx_status` IS NOT NULL AND lower(trim(`tx_status`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tx_status`) RLIKE '^0+([.,]0+)?$'),
    'st_simulacao', 'string', COUNT_IF(`st_simulacao` IS NULL), COUNT_IF(`st_simulacao` IS NOT NULL AND lower(trim(`st_simulacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_simulacao`) RLIKE '^0+([.,]0+)?$'),
    'tx_log', 'string', COUNT_IF(`tx_log` IS NULL), COUNT_IF(`tx_log` IS NOT NULL AND lower(trim(`tx_log`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tx_log`) RLIKE '^0+([.,]0+)?$'),
    'tx_log_simulacao', 'string', COUNT_IF(`tx_log_simulacao` IS NULL), COUNT_IF(`tx_log_simulacao` IS NOT NULL AND lower(trim(`tx_log_simulacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tx_log_simulacao`) RLIKE '^0+([.,]0+)?$'),
    'tx_justificativa', 'string', COUNT_IF(`tx_justificativa` IS NULL), COUNT_IF(`tx_justificativa` IS NOT NULL AND lower(trim(`tx_justificativa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tx_justificativa`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario', 'string', COUNT_IF(`nm_usuario` IS NULL), COUNT_IF(`nm_usuario` IS NOT NULL AND lower(trim(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario`) RLIKE '^0+([.,]0+)?$'),
    'cd_area_responsavel', 'string', COUNT_IF(`cd_area_responsavel` IS NULL), COUNT_IF(`cd_area_responsavel` IS NOT NULL AND lower(trim(`cd_area_responsavel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_area_responsavel`) RLIKE '^0+([.,]0+)?$'),
    'tp_documento_compras', 'string', COUNT_IF(`tp_documento_compras` IS NULL), COUNT_IF(`tp_documento_compras` IS NOT NULL AND lower(trim(`tp_documento_compras`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_documento_compras`) RLIKE '^0+([.,]0+)?$'),
    'tp_classificacao_contabil', 'string', COUNT_IF(`tp_classificacao_contabil` IS NULL), COUNT_IF(`tp_classificacao_contabil` IS NOT NULL AND lower(trim(`tp_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_classificacao_contabil`) RLIKE '^0+([.,]0+)?$'),
    'cd_grupo_mercadorias', 'string', COUNT_IF(`cd_grupo_mercadorias` IS NULL), COUNT_IF(`cd_grupo_mercadorias` IS NOT NULL AND lower(trim(`cd_grupo_mercadorias`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_grupo_mercadorias`) RLIKE '^0+([.,]0+)?$'),
    'dt_pedido_compra', 'date', COUNT_IF(`dt_pedido_compra` IS NULL), 0L, 0L,
    'vl_preco_liquido_pedido', 'double', COUNT_IF(`vl_preco_liquido_pedido` IS NULL), 0L, COUNT_IF(`vl_preco_liquido_pedido` = 0),
    'nr_remessa_entrega', 'string', COUNT_IF(`nr_remessa_entrega` IS NULL), COUNT_IF(`nr_remessa_entrega` IS NOT NULL AND lower(trim(`nr_remessa_entrega`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_remessa_entrega`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao_remessa', 'date', COUNT_IF(`dt_criacao_remessa` IS NULL), 0L, 0L,
    'hr_criacao_remessa', 'string', COUNT_IF(`hr_criacao_remessa` IS NULL), COUNT_IF(`hr_criacao_remessa` IS NOT NULL AND lower(trim(`hr_criacao_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`hr_criacao_remessa`) RLIKE '^0+([.,]0+)?$'),
    'nr_ordem', 'string', COUNT_IF(`nr_ordem` IS NULL), COUNT_IF(`nr_ordem` IS NOT NULL AND lower(trim(`nr_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nr_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cd_revisao_manutencao', 'string', COUNT_IF(`cd_revisao_manutencao` IS NULL), COUNT_IF(`cd_revisao_manutencao` IS NOT NULL AND lower(trim(`cd_revisao_manutencao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cd_revisao_manutencao`) RLIKE '^0+([.,]0+)?$'),
    'tp_mrp', 'string', COUNT_IF(`tp_mrp` IS NULL), COUNT_IF(`tp_mrp` IS NOT NULL AND lower(trim(`tp_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_mrp`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125),
card AS (
  SELECT stack(106,
    'nr_nota_fiscal_eletronica', 'string', approx_count_distinct(`nr_nota_fiscal_eletronica`),
    'cd_serie', 'string', approx_count_distinct(`cd_serie`),
    'cd_empresa', 'string', approx_count_distinct(`cd_empresa`),
    'cd_local_negocios', 'string', approx_count_distinct(`cd_local_negocios`),
    'cd_fornecedor', 'string', approx_count_distinct(`cd_fornecedor`),
    'cd_chave_acesso_nfe', 'string', approx_count_distinct(`cd_chave_acesso_nfe`),
    'nr_item_nota_fiscal', 'string', approx_count_distinct(`nr_item_nota_fiscal`),
    'cd_material', 'string', approx_count_distinct(`cd_material`),
    'cd_material_xml', 'string', approx_count_distinct(`cd_material_xml`),
    'nr_documento_compras', 'string', approx_count_distinct(`nr_documento_compras`),
    'nr_item_pedido', 'string', approx_count_distinct(`nr_item_pedido`),
    'nr_documento_faturamento', 'string', approx_count_distinct(`nr_documento_faturamento`),
    'nr_exercicio', 'string', approx_count_distinct(`nr_exercicio`),
    'nr_documento_material', 'string', approx_count_distinct(`nr_documento_material`),
    'nr_remessa', 'string', approx_count_distinct(`nr_remessa`),
    'cd_chave_acesso_dacte', 'string', approx_count_distinct(`cd_chave_acesso_dacte`),
    'cd_cnpj_transportador', 'string', approx_count_distinct(`cd_cnpj_transportador`),
    'cd_cnpj', 'string', approx_count_distinct(`cd_cnpj`),
    'cd_codigo_barras', 'string', approx_count_distinct(`cd_codigo_barras`),
    'qt_contagem_cte', 'int', approx_count_distinct(`qt_contagem_cte`),
    'nr_transporte', 'string', approx_count_distinct(`nr_transporte`),
    'cd_centro', 'string', approx_count_distinct(`cd_centro`),
    'nm_centro', 'string', approx_count_distinct(`nm_centro`),
    'dt_documento', 'date', approx_count_distinct(`dt_documento`),
    'dt_recebimento', 'date', approx_count_distinct(`dt_recebimento`),
    'dt_recebimento_item', 'date', approx_count_distinct(`dt_recebimento_item`),
    'dt_justificativa', 'date', approx_count_distinct(`dt_justificativa`),
    'dt_data', 'date', approx_count_distinct(`dt_data`),
    'dt_lancamento', 'date', approx_count_distinct(`dt_lancamento`),
    'dt_vencimento_duplicata', 'date', approx_count_distinct(`dt_vencimento_duplicata`),
    'hr_hora', 'string', approx_count_distinct(`hr_hora`),
    'cd_ncm', 'string', approx_count_distinct(`cd_ncm`),
    'cd_cfop', 'string', approx_count_distinct(`cd_cfop`),
    'cd_cst_icms', 'string', approx_count_distinct(`cd_cst_icms`),
    'cd_iva', 'string', approx_count_distinct(`cd_iva`),
    'cd_origem_material', 'string', approx_count_distinct(`cd_origem_material`),
    'cd_procedencia_material', 'string', approx_count_distinct(`cd_procedencia_material`),
    'tx_descricao_produto', 'string', approx_count_distinct(`tx_descricao_produto`),
    'tx_descricao_material', 'string', approx_count_distinct(`tx_descricao_material`),
    'qt_quantidade', 'decimal(13,3)', approx_count_distinct(`qt_quantidade`),
    'qt_quantidade_xml', 'decimal(13,3)', approx_count_distinct(`qt_quantidade_xml`),
    'qt_quantidade_recebida', 'decimal(13,3)', approx_count_distinct(`qt_quantidade_recebida`),
    'cd_unidade_medida', 'string', approx_count_distinct(`cd_unidade_medida`),
    'cd_unidade_medida_xml', 'string', approx_count_distinct(`cd_unidade_medida_xml`),
    'cd_unidade_medida_recebimento', 'string', approx_count_distinct(`cd_unidade_medida_recebimento`),
    'cd_unidade_medida_nfe', 'string', approx_count_distinct(`cd_unidade_medida_nfe`),
    'vl_preco_unitario_bruto', 'double', approx_count_distinct(`vl_preco_unitario_bruto`),
    'vl_preco_unitario_liquido', 'double', approx_count_distinct(`vl_preco_unitario_liquido`),
    'vl_unitario', 'double', approx_count_distinct(`vl_unitario`),
    'vl_bruto', 'double', approx_count_distinct(`vl_bruto`),
    'vl_liquido', 'double', approx_count_distinct(`vl_liquido`),
    'vl_total', 'double', approx_count_distinct(`vl_total`),
    'pc_aliquota_icms', 'decimal(7,2)', approx_count_distinct(`pc_aliquota_icms`),
    'vl_base_icms', 'double', approx_count_distinct(`vl_base_icms`),
    'vl_icms', 'double', approx_count_distinct(`vl_icms`),
    'pc_aliquota_st', 'decimal(7,2)', approx_count_distinct(`pc_aliquota_st`),
    'vl_base_st', 'double', approx_count_distinct(`vl_base_st`),
    'vl_st', 'double', approx_count_distinct(`vl_st`),
    'pc_aliquota_ipi', 'decimal(7,2)', approx_count_distinct(`pc_aliquota_ipi`),
    'vl_base_ipi', 'double', approx_count_distinct(`vl_base_ipi`),
    'vl_ipi', 'double', approx_count_distinct(`vl_ipi`),
    'pc_aliquota_pis', 'decimal(7,2)', approx_count_distinct(`pc_aliquota_pis`),
    'vl_base_pis', 'double', approx_count_distinct(`vl_base_pis`),
    'vl_pis', 'double', approx_count_distinct(`vl_pis`),
    'pc_aliquota_cofins', 'decimal(7,2)', approx_count_distinct(`pc_aliquota_cofins`),
    'vl_base_cofins', 'double', approx_count_distinct(`vl_base_cofins`),
    'vl_cofins', 'double', approx_count_distinct(`vl_cofins`),
    'vl_base_calculo_icms_item', 'double', approx_count_distinct(`vl_base_calculo_icms_item`),
    'vl_total_bruto', 'double', approx_count_distinct(`vl_total_bruto`),
    'vl_total_liquido', 'double', approx_count_distinct(`vl_total_liquido`),
    'vl_total_icms', 'double', approx_count_distinct(`vl_total_icms`),
    'vl_total_st', 'double', approx_count_distinct(`vl_total_st`),
    'vl_total_ipi', 'double', approx_count_distinct(`vl_total_ipi`),
    'vl_total_produto', 'double', approx_count_distinct(`vl_total_produto`),
    'vl_despesas_acessorias', 'double', approx_count_distinct(`vl_despesas_acessorias`),
    'vl_frete', 'double', approx_count_distinct(`vl_frete`),
    'vl_icms_nao_destacado', 'double', approx_count_distinct(`vl_icms_nao_destacado`),
    'vl_icms_st_nao_destacado', 'double', approx_count_distinct(`vl_icms_st_nao_destacado`),
    'vl_base_calculo_icms', 'double', approx_count_distinct(`vl_base_calculo_icms`),
    'vl_base_calculo_st', 'double', approx_count_distinct(`vl_base_calculo_st`),
    'qt_peso_total_nota_fiscal', 'decimal(15,3)', approx_count_distinct(`qt_peso_total_nota_fiscal`),
    'qt_peso_liquido', 'decimal(15,3)', approx_count_distinct(`qt_peso_liquido`),
    'cd_unidade_peso', 'string', approx_count_distinct(`cd_unidade_peso`),
    'qt_volumes', 'string', approx_count_distinct(`qt_volumes`),
    'cd_incoterms', 'string', approx_count_distinct(`cd_incoterms`),
    'cd_modalidade_frete', 'string', approx_count_distinct(`cd_modalidade_frete`),
    'nm_transportador', 'string', approx_count_distinct(`nm_transportador`),
    'cd_frente', 'string', approx_count_distinct(`cd_frente`),
    'tx_status', 'string', approx_count_distinct(`tx_status`),
    'st_simulacao', 'string', approx_count_distinct(`st_simulacao`),
    'tx_log', 'string', approx_count_distinct(`tx_log`),
    'tx_log_simulacao', 'string', approx_count_distinct(`tx_log_simulacao`),
    'tx_justificativa', 'string', approx_count_distinct(`tx_justificativa`),
    'nm_usuario', 'string', approx_count_distinct(`nm_usuario`),
    'cd_area_responsavel', 'string', approx_count_distinct(`cd_area_responsavel`),
    'tp_documento_compras', 'string', approx_count_distinct(`tp_documento_compras`),
    'tp_classificacao_contabil', 'string', approx_count_distinct(`tp_classificacao_contabil`),
    'cd_grupo_mercadorias', 'string', approx_count_distinct(`cd_grupo_mercadorias`),
    'dt_pedido_compra', 'date', approx_count_distinct(`dt_pedido_compra`),
    'vl_preco_liquido_pedido', 'double', approx_count_distinct(`vl_preco_liquido_pedido`),
    'nr_remessa_entrega', 'string', approx_count_distinct(`nr_remessa_entrega`),
    'dt_criacao_remessa', 'date', approx_count_distinct(`dt_criacao_remessa`),
    'hr_criacao_remessa', 'string', approx_count_distinct(`hr_criacao_remessa`),
    'nr_ordem', 'string', approx_count_distinct(`nr_ordem`),
    'cd_revisao_manutencao', 'string', approx_count_distinct(`cd_revisao_manutencao`),
    'tp_mrp', 'string', approx_count_distinct(`tp_mrp`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS (top 8 de cada)
(SELECT 'cd_serie' AS coluna, CAST(`cd_serie` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_serie` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_empresa' AS coluna, CAST(`cd_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_local_negocios' AS coluna, CAST(`cd_local_negocios` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_local_negocios` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_centro' AS coluna, CAST(`cd_centro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_cfop' AS coluna, CAST(`cd_cfop` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_cfop` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_cst_icms' AS coluna, CAST(`cd_cst_icms` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_cst_icms` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_iva' AS coluna, CAST(`cd_iva` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_iva` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_origem_material' AS coluna, CAST(`cd_origem_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_origem_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_procedencia_material' AS coluna, CAST(`cd_procedencia_material` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_procedencia_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_unidade_medida' AS coluna, CAST(`cd_unidade_medida` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_unidade_medida` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_modalidade_frete' AS coluna, CAST(`cd_modalidade_frete` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_modalidade_frete` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_incoterms' AS coluna, CAST(`cd_incoterms` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_incoterms` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tx_status' AS coluna, CAST(`tx_status` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `tx_status` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_simulacao' AS coluna, CAST(`st_simulacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `st_simulacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_area_responsavel' AS coluna, CAST(`cd_area_responsavel` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_area_responsavel` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_documento_compras' AS coluna, CAST(`tp_documento_compras` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `tp_documento_compras` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_classificacao_contabil' AS coluna, CAST(`tp_classificacao_contabil` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `tp_classificacao_contabil` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_grupo_mercadorias' AS coluna, CAST(`cd_grupo_mercadorias` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_grupo_mercadorias` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_mrp' AS coluna, CAST(`tp_mrp` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `tp_mrp` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cd_revisao_manutencao' AS coluna, CAST(`cd_revisao_manutencao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125 GROUP BY `cd_revisao_manutencao` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil dos campos numéricos

Entre as colunas analisadas, **26 são `double`** — exigem tolerância de 0,005 na comparação com o SAP.

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(37,
    'qt_quantidade', 'decimal(13,3)', COUNT(`qt_quantidade`), CAST(MIN(`qt_quantidade`) AS DOUBLE), CAST(MAX(`qt_quantidade`) AS DOUBLE), CAST(AVG(`qt_quantidade`) AS DOUBLE), CAST(percentile_approx(`qt_quantidade`, 0.5) AS DOUBLE), COUNT_IF(`qt_quantidade` < 0), COUNT_IF(`qt_quantidade` = 0),
    'qt_quantidade_xml', 'decimal(13,3)', COUNT(`qt_quantidade_xml`), CAST(MIN(`qt_quantidade_xml`) AS DOUBLE), CAST(MAX(`qt_quantidade_xml`) AS DOUBLE), CAST(AVG(`qt_quantidade_xml`) AS DOUBLE), CAST(percentile_approx(`qt_quantidade_xml`, 0.5) AS DOUBLE), COUNT_IF(`qt_quantidade_xml` < 0), COUNT_IF(`qt_quantidade_xml` = 0),
    'qt_quantidade_recebida', 'decimal(13,3)', COUNT(`qt_quantidade_recebida`), CAST(MIN(`qt_quantidade_recebida`) AS DOUBLE), CAST(MAX(`qt_quantidade_recebida`) AS DOUBLE), CAST(AVG(`qt_quantidade_recebida`) AS DOUBLE), CAST(percentile_approx(`qt_quantidade_recebida`, 0.5) AS DOUBLE), COUNT_IF(`qt_quantidade_recebida` < 0), COUNT_IF(`qt_quantidade_recebida` = 0),
    'vl_preco_unitario_bruto', 'double', COUNT(`vl_preco_unitario_bruto`), CAST(MIN(`vl_preco_unitario_bruto`) AS DOUBLE), CAST(MAX(`vl_preco_unitario_bruto`) AS DOUBLE), CAST(AVG(`vl_preco_unitario_bruto`) AS DOUBLE), CAST(percentile_approx(`vl_preco_unitario_bruto`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_unitario_bruto` < 0), COUNT_IF(`vl_preco_unitario_bruto` = 0),
    'vl_preco_unitario_liquido', 'double', COUNT(`vl_preco_unitario_liquido`), CAST(MIN(`vl_preco_unitario_liquido`) AS DOUBLE), CAST(MAX(`vl_preco_unitario_liquido`) AS DOUBLE), CAST(AVG(`vl_preco_unitario_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_preco_unitario_liquido`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_unitario_liquido` < 0), COUNT_IF(`vl_preco_unitario_liquido` = 0),
    'vl_unitario', 'double', COUNT(`vl_unitario`), CAST(MIN(`vl_unitario`) AS DOUBLE), CAST(MAX(`vl_unitario`) AS DOUBLE), CAST(AVG(`vl_unitario`) AS DOUBLE), CAST(percentile_approx(`vl_unitario`, 0.5) AS DOUBLE), COUNT_IF(`vl_unitario` < 0), COUNT_IF(`vl_unitario` = 0),
    'vl_bruto', 'double', COUNT(`vl_bruto`), CAST(MIN(`vl_bruto`) AS DOUBLE), CAST(MAX(`vl_bruto`) AS DOUBLE), CAST(AVG(`vl_bruto`) AS DOUBLE), CAST(percentile_approx(`vl_bruto`, 0.5) AS DOUBLE), COUNT_IF(`vl_bruto` < 0), COUNT_IF(`vl_bruto` = 0),
    'vl_liquido', 'double', COUNT(`vl_liquido`), CAST(MIN(`vl_liquido`) AS DOUBLE), CAST(MAX(`vl_liquido`) AS DOUBLE), CAST(AVG(`vl_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_liquido`, 0.5) AS DOUBLE), COUNT_IF(`vl_liquido` < 0), COUNT_IF(`vl_liquido` = 0),
    'vl_total', 'double', COUNT(`vl_total`), CAST(MIN(`vl_total`) AS DOUBLE), CAST(MAX(`vl_total`) AS DOUBLE), CAST(AVG(`vl_total`) AS DOUBLE), CAST(percentile_approx(`vl_total`, 0.5) AS DOUBLE), COUNT_IF(`vl_total` < 0), COUNT_IF(`vl_total` = 0),
    'pc_aliquota_icms', 'decimal(7,2)', COUNT(`pc_aliquota_icms`), CAST(MIN(`pc_aliquota_icms`) AS DOUBLE), CAST(MAX(`pc_aliquota_icms`) AS DOUBLE), CAST(AVG(`pc_aliquota_icms`) AS DOUBLE), CAST(percentile_approx(`pc_aliquota_icms`, 0.5) AS DOUBLE), COUNT_IF(`pc_aliquota_icms` < 0), COUNT_IF(`pc_aliquota_icms` = 0),
    'vl_base_icms', 'double', COUNT(`vl_base_icms`), CAST(MIN(`vl_base_icms`) AS DOUBLE), CAST(MAX(`vl_base_icms`) AS DOUBLE), CAST(AVG(`vl_base_icms`) AS DOUBLE), CAST(percentile_approx(`vl_base_icms`, 0.5) AS DOUBLE), COUNT_IF(`vl_base_icms` < 0), COUNT_IF(`vl_base_icms` = 0),
    'vl_icms', 'double', COUNT(`vl_icms`), CAST(MIN(`vl_icms`) AS DOUBLE), CAST(MAX(`vl_icms`) AS DOUBLE), CAST(AVG(`vl_icms`) AS DOUBLE), CAST(percentile_approx(`vl_icms`, 0.5) AS DOUBLE), COUNT_IF(`vl_icms` < 0), COUNT_IF(`vl_icms` = 0),
    'pc_aliquota_st', 'decimal(7,2)', COUNT(`pc_aliquota_st`), CAST(MIN(`pc_aliquota_st`) AS DOUBLE), CAST(MAX(`pc_aliquota_st`) AS DOUBLE), CAST(AVG(`pc_aliquota_st`) AS DOUBLE), CAST(percentile_approx(`pc_aliquota_st`, 0.5) AS DOUBLE), COUNT_IF(`pc_aliquota_st` < 0), COUNT_IF(`pc_aliquota_st` = 0),
    'vl_base_st', 'double', COUNT(`vl_base_st`), CAST(MIN(`vl_base_st`) AS DOUBLE), CAST(MAX(`vl_base_st`) AS DOUBLE), CAST(AVG(`vl_base_st`) AS DOUBLE), CAST(percentile_approx(`vl_base_st`, 0.5) AS DOUBLE), COUNT_IF(`vl_base_st` < 0), COUNT_IF(`vl_base_st` = 0),
    'vl_st', 'double', COUNT(`vl_st`), CAST(MIN(`vl_st`) AS DOUBLE), CAST(MAX(`vl_st`) AS DOUBLE), CAST(AVG(`vl_st`) AS DOUBLE), CAST(percentile_approx(`vl_st`, 0.5) AS DOUBLE), COUNT_IF(`vl_st` < 0), COUNT_IF(`vl_st` = 0),
    'pc_aliquota_ipi', 'decimal(7,2)', COUNT(`pc_aliquota_ipi`), CAST(MIN(`pc_aliquota_ipi`) AS DOUBLE), CAST(MAX(`pc_aliquota_ipi`) AS DOUBLE), CAST(AVG(`pc_aliquota_ipi`) AS DOUBLE), CAST(percentile_approx(`pc_aliquota_ipi`, 0.5) AS DOUBLE), COUNT_IF(`pc_aliquota_ipi` < 0), COUNT_IF(`pc_aliquota_ipi` = 0),
    'vl_base_ipi', 'double', COUNT(`vl_base_ipi`), CAST(MIN(`vl_base_ipi`) AS DOUBLE), CAST(MAX(`vl_base_ipi`) AS DOUBLE), CAST(AVG(`vl_base_ipi`) AS DOUBLE), CAST(percentile_approx(`vl_base_ipi`, 0.5) AS DOUBLE), COUNT_IF(`vl_base_ipi` < 0), COUNT_IF(`vl_base_ipi` = 0),
    'vl_ipi', 'double', COUNT(`vl_ipi`), CAST(MIN(`vl_ipi`) AS DOUBLE), CAST(MAX(`vl_ipi`) AS DOUBLE), CAST(AVG(`vl_ipi`) AS DOUBLE), CAST(percentile_approx(`vl_ipi`, 0.5) AS DOUBLE), COUNT_IF(`vl_ipi` < 0), COUNT_IF(`vl_ipi` = 0),
    'pc_aliquota_pis', 'decimal(7,2)', COUNT(`pc_aliquota_pis`), CAST(MIN(`pc_aliquota_pis`) AS DOUBLE), CAST(MAX(`pc_aliquota_pis`) AS DOUBLE), CAST(AVG(`pc_aliquota_pis`) AS DOUBLE), CAST(percentile_approx(`pc_aliquota_pis`, 0.5) AS DOUBLE), COUNT_IF(`pc_aliquota_pis` < 0), COUNT_IF(`pc_aliquota_pis` = 0),
    'vl_base_pis', 'double', COUNT(`vl_base_pis`), CAST(MIN(`vl_base_pis`) AS DOUBLE), CAST(MAX(`vl_base_pis`) AS DOUBLE), CAST(AVG(`vl_base_pis`) AS DOUBLE), CAST(percentile_approx(`vl_base_pis`, 0.5) AS DOUBLE), COUNT_IF(`vl_base_pis` < 0), COUNT_IF(`vl_base_pis` = 0),
    'vl_pis', 'double', COUNT(`vl_pis`), CAST(MIN(`vl_pis`) AS DOUBLE), CAST(MAX(`vl_pis`) AS DOUBLE), CAST(AVG(`vl_pis`) AS DOUBLE), CAST(percentile_approx(`vl_pis`, 0.5) AS DOUBLE), COUNT_IF(`vl_pis` < 0), COUNT_IF(`vl_pis` = 0),
    'pc_aliquota_cofins', 'decimal(7,2)', COUNT(`pc_aliquota_cofins`), CAST(MIN(`pc_aliquota_cofins`) AS DOUBLE), CAST(MAX(`pc_aliquota_cofins`) AS DOUBLE), CAST(AVG(`pc_aliquota_cofins`) AS DOUBLE), CAST(percentile_approx(`pc_aliquota_cofins`, 0.5) AS DOUBLE), COUNT_IF(`pc_aliquota_cofins` < 0), COUNT_IF(`pc_aliquota_cofins` = 0),
    'vl_base_cofins', 'double', COUNT(`vl_base_cofins`), CAST(MIN(`vl_base_cofins`) AS DOUBLE), CAST(MAX(`vl_base_cofins`) AS DOUBLE), CAST(AVG(`vl_base_cofins`) AS DOUBLE), CAST(percentile_approx(`vl_base_cofins`, 0.5) AS DOUBLE), COUNT_IF(`vl_base_cofins` < 0), COUNT_IF(`vl_base_cofins` = 0),
    'vl_cofins', 'double', COUNT(`vl_cofins`), CAST(MIN(`vl_cofins`) AS DOUBLE), CAST(MAX(`vl_cofins`) AS DOUBLE), CAST(AVG(`vl_cofins`) AS DOUBLE), CAST(percentile_approx(`vl_cofins`, 0.5) AS DOUBLE), COUNT_IF(`vl_cofins` < 0), COUNT_IF(`vl_cofins` = 0),
    'vl_base_calculo_icms_item', 'double', COUNT(`vl_base_calculo_icms_item`), CAST(MIN(`vl_base_calculo_icms_item`) AS DOUBLE), CAST(MAX(`vl_base_calculo_icms_item`) AS DOUBLE), CAST(AVG(`vl_base_calculo_icms_item`) AS DOUBLE), CAST(percentile_approx(`vl_base_calculo_icms_item`, 0.5) AS DOUBLE), COUNT_IF(`vl_base_calculo_icms_item` < 0), COUNT_IF(`vl_base_calculo_icms_item` = 0),
    'vl_total_bruto', 'double', COUNT(`vl_total_bruto`), CAST(MIN(`vl_total_bruto`) AS DOUBLE), CAST(MAX(`vl_total_bruto`) AS DOUBLE), CAST(AVG(`vl_total_bruto`) AS DOUBLE), CAST(percentile_approx(`vl_total_bruto`, 0.5) AS DOUBLE), COUNT_IF(`vl_total_bruto` < 0), COUNT_IF(`vl_total_bruto` = 0),
    'vl_total_liquido', 'double', COUNT(`vl_total_liquido`), CAST(MIN(`vl_total_liquido`) AS DOUBLE), CAST(MAX(`vl_total_liquido`) AS DOUBLE), CAST(AVG(`vl_total_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_total_liquido`, 0.5) AS DOUBLE), COUNT_IF(`vl_total_liquido` < 0), COUNT_IF(`vl_total_liquido` = 0),
    'vl_total_icms', 'double', COUNT(`vl_total_icms`), CAST(MIN(`vl_total_icms`) AS DOUBLE), CAST(MAX(`vl_total_icms`) AS DOUBLE), CAST(AVG(`vl_total_icms`) AS DOUBLE), CAST(percentile_approx(`vl_total_icms`, 0.5) AS DOUBLE), COUNT_IF(`vl_total_icms` < 0), COUNT_IF(`vl_total_icms` = 0),
    'vl_total_st', 'double', COUNT(`vl_total_st`), CAST(MIN(`vl_total_st`) AS DOUBLE), CAST(MAX(`vl_total_st`) AS DOUBLE), CAST(AVG(`vl_total_st`) AS DOUBLE), CAST(percentile_approx(`vl_total_st`, 0.5) AS DOUBLE), COUNT_IF(`vl_total_st` < 0), COUNT_IF(`vl_total_st` = 0),
    'vl_total_ipi', 'double', COUNT(`vl_total_ipi`), CAST(MIN(`vl_total_ipi`) AS DOUBLE), CAST(MAX(`vl_total_ipi`) AS DOUBLE), CAST(AVG(`vl_total_ipi`) AS DOUBLE), CAST(percentile_approx(`vl_total_ipi`, 0.5) AS DOUBLE), COUNT_IF(`vl_total_ipi` < 0), COUNT_IF(`vl_total_ipi` = 0),
    'vl_total_produto', 'double', COUNT(`vl_total_produto`), CAST(MIN(`vl_total_produto`) AS DOUBLE), CAST(MAX(`vl_total_produto`) AS DOUBLE), CAST(AVG(`vl_total_produto`) AS DOUBLE), CAST(percentile_approx(`vl_total_produto`, 0.5) AS DOUBLE), COUNT_IF(`vl_total_produto` < 0), COUNT_IF(`vl_total_produto` = 0),
    'vl_despesas_acessorias', 'double', COUNT(`vl_despesas_acessorias`), CAST(MIN(`vl_despesas_acessorias`) AS DOUBLE), CAST(MAX(`vl_despesas_acessorias`) AS DOUBLE), CAST(AVG(`vl_despesas_acessorias`) AS DOUBLE), CAST(percentile_approx(`vl_despesas_acessorias`, 0.5) AS DOUBLE), COUNT_IF(`vl_despesas_acessorias` < 0), COUNT_IF(`vl_despesas_acessorias` = 0),
    'vl_frete', 'double', COUNT(`vl_frete`), CAST(MIN(`vl_frete`) AS DOUBLE), CAST(MAX(`vl_frete`) AS DOUBLE), CAST(AVG(`vl_frete`) AS DOUBLE), CAST(percentile_approx(`vl_frete`, 0.5) AS DOUBLE), COUNT_IF(`vl_frete` < 0), COUNT_IF(`vl_frete` = 0),
    'qt_peso_total_nota_fiscal', 'decimal(15,3)', COUNT(`qt_peso_total_nota_fiscal`), CAST(MIN(`qt_peso_total_nota_fiscal`) AS DOUBLE), CAST(MAX(`qt_peso_total_nota_fiscal`) AS DOUBLE), CAST(AVG(`qt_peso_total_nota_fiscal`) AS DOUBLE), CAST(percentile_approx(`qt_peso_total_nota_fiscal`, 0.5) AS DOUBLE), COUNT_IF(`qt_peso_total_nota_fiscal` < 0), COUNT_IF(`qt_peso_total_nota_fiscal` = 0),
    'qt_peso_liquido', 'decimal(15,3)', COUNT(`qt_peso_liquido`), CAST(MIN(`qt_peso_liquido`) AS DOUBLE), CAST(MAX(`qt_peso_liquido`) AS DOUBLE), CAST(AVG(`qt_peso_liquido`) AS DOUBLE), CAST(percentile_approx(`qt_peso_liquido`, 0.5) AS DOUBLE), COUNT_IF(`qt_peso_liquido` < 0), COUNT_IF(`qt_peso_liquido` = 0),
    'vl_preco_liquido_pedido', 'double', COUNT(`vl_preco_liquido_pedido`), CAST(MIN(`vl_preco_liquido_pedido`) AS DOUBLE), CAST(MAX(`vl_preco_liquido_pedido`) AS DOUBLE), CAST(AVG(`vl_preco_liquido_pedido`) AS DOUBLE), CAST(percentile_approx(`vl_preco_liquido_pedido`, 0.5) AS DOUBLE), COUNT_IF(`vl_preco_liquido_pedido` < 0), COUNT_IF(`vl_preco_liquido_pedido` = 0),
    'qt_contagem_cte', 'int', COUNT(`qt_contagem_cte`), CAST(MIN(`qt_contagem_cte`) AS DOUBLE), CAST(MAX(`qt_contagem_cte`) AS DOUBLE), CAST(AVG(`qt_contagem_cte`) AS DOUBLE), CAST(percentile_approx(`qt_contagem_cte`, 0.5) AS DOUBLE), COUNT_IF(`qt_contagem_cte` < 0), COUNT_IF(`qt_contagem_cte` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(37,
    'qt_quantidade', CAST(SUM(`qt_quantidade`) AS DOUBLE), CAST(ROUND(SUM(`qt_quantidade`), 2) AS STRING), COUNT(`qt_quantidade`),
    'qt_quantidade_xml', CAST(SUM(`qt_quantidade_xml`) AS DOUBLE), CAST(ROUND(SUM(`qt_quantidade_xml`), 2) AS STRING), COUNT(`qt_quantidade_xml`),
    'qt_quantidade_recebida', CAST(SUM(`qt_quantidade_recebida`) AS DOUBLE), CAST(ROUND(SUM(`qt_quantidade_recebida`), 2) AS STRING), COUNT(`qt_quantidade_recebida`),
    'vl_preco_unitario_bruto', CAST(SUM(`vl_preco_unitario_bruto`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_unitario_bruto`), 2) AS STRING), COUNT(`vl_preco_unitario_bruto`),
    'vl_preco_unitario_liquido', CAST(SUM(`vl_preco_unitario_liquido`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_unitario_liquido`), 2) AS STRING), COUNT(`vl_preco_unitario_liquido`),
    'vl_unitario', CAST(SUM(`vl_unitario`) AS DOUBLE), CAST(ROUND(SUM(`vl_unitario`), 2) AS STRING), COUNT(`vl_unitario`),
    'vl_bruto', CAST(SUM(`vl_bruto`) AS DOUBLE), CAST(ROUND(SUM(`vl_bruto`), 2) AS STRING), COUNT(`vl_bruto`),
    'vl_liquido', CAST(SUM(`vl_liquido`) AS DOUBLE), CAST(ROUND(SUM(`vl_liquido`), 2) AS STRING), COUNT(`vl_liquido`),
    'vl_total', CAST(SUM(`vl_total`) AS DOUBLE), CAST(ROUND(SUM(`vl_total`), 2) AS STRING), COUNT(`vl_total`),
    'pc_aliquota_icms', CAST(SUM(`pc_aliquota_icms`) AS DOUBLE), CAST(ROUND(SUM(`pc_aliquota_icms`), 2) AS STRING), COUNT(`pc_aliquota_icms`),
    'vl_base_icms', CAST(SUM(`vl_base_icms`) AS DOUBLE), CAST(ROUND(SUM(`vl_base_icms`), 2) AS STRING), COUNT(`vl_base_icms`),
    'vl_icms', CAST(SUM(`vl_icms`) AS DOUBLE), CAST(ROUND(SUM(`vl_icms`), 2) AS STRING), COUNT(`vl_icms`),
    'pc_aliquota_st', CAST(SUM(`pc_aliquota_st`) AS DOUBLE), CAST(ROUND(SUM(`pc_aliquota_st`), 2) AS STRING), COUNT(`pc_aliquota_st`),
    'vl_base_st', CAST(SUM(`vl_base_st`) AS DOUBLE), CAST(ROUND(SUM(`vl_base_st`), 2) AS STRING), COUNT(`vl_base_st`),
    'vl_st', CAST(SUM(`vl_st`) AS DOUBLE), CAST(ROUND(SUM(`vl_st`), 2) AS STRING), COUNT(`vl_st`),
    'pc_aliquota_ipi', CAST(SUM(`pc_aliquota_ipi`) AS DOUBLE), CAST(ROUND(SUM(`pc_aliquota_ipi`), 2) AS STRING), COUNT(`pc_aliquota_ipi`),
    'vl_base_ipi', CAST(SUM(`vl_base_ipi`) AS DOUBLE), CAST(ROUND(SUM(`vl_base_ipi`), 2) AS STRING), COUNT(`vl_base_ipi`),
    'vl_ipi', CAST(SUM(`vl_ipi`) AS DOUBLE), CAST(ROUND(SUM(`vl_ipi`), 2) AS STRING), COUNT(`vl_ipi`),
    'pc_aliquota_pis', CAST(SUM(`pc_aliquota_pis`) AS DOUBLE), CAST(ROUND(SUM(`pc_aliquota_pis`), 2) AS STRING), COUNT(`pc_aliquota_pis`),
    'vl_base_pis', CAST(SUM(`vl_base_pis`) AS DOUBLE), CAST(ROUND(SUM(`vl_base_pis`), 2) AS STRING), COUNT(`vl_base_pis`),
    'vl_pis', CAST(SUM(`vl_pis`) AS DOUBLE), CAST(ROUND(SUM(`vl_pis`), 2) AS STRING), COUNT(`vl_pis`),
    'pc_aliquota_cofins', CAST(SUM(`pc_aliquota_cofins`) AS DOUBLE), CAST(ROUND(SUM(`pc_aliquota_cofins`), 2) AS STRING), COUNT(`pc_aliquota_cofins`),
    'vl_base_cofins', CAST(SUM(`vl_base_cofins`) AS DOUBLE), CAST(ROUND(SUM(`vl_base_cofins`), 2) AS STRING), COUNT(`vl_base_cofins`),
    'vl_cofins', CAST(SUM(`vl_cofins`) AS DOUBLE), CAST(ROUND(SUM(`vl_cofins`), 2) AS STRING), COUNT(`vl_cofins`),
    'vl_base_calculo_icms_item', CAST(SUM(`vl_base_calculo_icms_item`) AS DOUBLE), CAST(ROUND(SUM(`vl_base_calculo_icms_item`), 2) AS STRING), COUNT(`vl_base_calculo_icms_item`),
    'vl_total_bruto', CAST(SUM(`vl_total_bruto`) AS DOUBLE), CAST(ROUND(SUM(`vl_total_bruto`), 2) AS STRING), COUNT(`vl_total_bruto`),
    'vl_total_liquido', CAST(SUM(`vl_total_liquido`) AS DOUBLE), CAST(ROUND(SUM(`vl_total_liquido`), 2) AS STRING), COUNT(`vl_total_liquido`),
    'vl_total_icms', CAST(SUM(`vl_total_icms`) AS DOUBLE), CAST(ROUND(SUM(`vl_total_icms`), 2) AS STRING), COUNT(`vl_total_icms`),
    'vl_total_st', CAST(SUM(`vl_total_st`) AS DOUBLE), CAST(ROUND(SUM(`vl_total_st`), 2) AS STRING), COUNT(`vl_total_st`),
    'vl_total_ipi', CAST(SUM(`vl_total_ipi`) AS DOUBLE), CAST(ROUND(SUM(`vl_total_ipi`), 2) AS STRING), COUNT(`vl_total_ipi`),
    'vl_total_produto', CAST(SUM(`vl_total_produto`) AS DOUBLE), CAST(ROUND(SUM(`vl_total_produto`), 2) AS STRING), COUNT(`vl_total_produto`),
    'vl_despesas_acessorias', CAST(SUM(`vl_despesas_acessorias`) AS DOUBLE), CAST(ROUND(SUM(`vl_despesas_acessorias`), 2) AS STRING), COUNT(`vl_despesas_acessorias`),
    'vl_frete', CAST(SUM(`vl_frete`) AS DOUBLE), CAST(ROUND(SUM(`vl_frete`), 2) AS STRING), COUNT(`vl_frete`),
    'qt_peso_total_nota_fiscal', CAST(SUM(`qt_peso_total_nota_fiscal`) AS DOUBLE), CAST(ROUND(SUM(`qt_peso_total_nota_fiscal`), 2) AS STRING), COUNT(`qt_peso_total_nota_fiscal`),
    'qt_peso_liquido', CAST(SUM(`qt_peso_liquido`) AS DOUBLE), CAST(ROUND(SUM(`qt_peso_liquido`), 2) AS STRING), COUNT(`qt_peso_liquido`),
    'vl_preco_liquido_pedido', CAST(SUM(`vl_preco_liquido_pedido`) AS DOUBLE), CAST(ROUND(SUM(`vl_preco_liquido_pedido`), 2) AS STRING), COUNT(`vl_preco_liquido_pedido`),
    'qt_contagem_cte', CAST(SUM(`qt_contagem_cte`) AS DOUBLE), CAST(ROUND(SUM(`qt_contagem_cte`), 2) AS STRING), COUNT(`qt_contagem_cte`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
)
ORDER BY coluna;

## 12. Datas

Todas as datas desta tabela estão em tipo nativo `date`. A coluna `sentinela_9999`
conta ocorrências de `9999-12-31`, usada pelo SAP para "sem prazo definido".

In [0]:
-- DATAS EM TIPO NATIVO
SELECT coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999
FROM (
  SELECT stack(9,
    'dt_documento', COUNT_IF(`dt_documento` IS NULL), CAST(MIN(`dt_documento`) AS STRING), CAST(MAX(`dt_documento`) AS STRING), COUNT(DISTINCT `dt_documento`), COUNT_IF(`dt_documento` > current_date()), COUNT_IF(`dt_documento` = DATE'9999-12-31'),
    'dt_recebimento', COUNT_IF(`dt_recebimento` IS NULL), CAST(MIN(`dt_recebimento`) AS STRING), CAST(MAX(`dt_recebimento`) AS STRING), COUNT(DISTINCT `dt_recebimento`), COUNT_IF(`dt_recebimento` > current_date()), COUNT_IF(`dt_recebimento` = DATE'9999-12-31'),
    'dt_recebimento_item', COUNT_IF(`dt_recebimento_item` IS NULL), CAST(MIN(`dt_recebimento_item`) AS STRING), CAST(MAX(`dt_recebimento_item`) AS STRING), COUNT(DISTINCT `dt_recebimento_item`), COUNT_IF(`dt_recebimento_item` > current_date()), COUNT_IF(`dt_recebimento_item` = DATE'9999-12-31'),
    'dt_justificativa', COUNT_IF(`dt_justificativa` IS NULL), CAST(MIN(`dt_justificativa`) AS STRING), CAST(MAX(`dt_justificativa`) AS STRING), COUNT(DISTINCT `dt_justificativa`), COUNT_IF(`dt_justificativa` > current_date()), COUNT_IF(`dt_justificativa` = DATE'9999-12-31'),
    'dt_data', COUNT_IF(`dt_data` IS NULL), CAST(MIN(`dt_data`) AS STRING), CAST(MAX(`dt_data`) AS STRING), COUNT(DISTINCT `dt_data`), COUNT_IF(`dt_data` > current_date()), COUNT_IF(`dt_data` = DATE'9999-12-31'),
    'dt_lancamento', COUNT_IF(`dt_lancamento` IS NULL), CAST(MIN(`dt_lancamento`) AS STRING), CAST(MAX(`dt_lancamento`) AS STRING), COUNT(DISTINCT `dt_lancamento`), COUNT_IF(`dt_lancamento` > current_date()), COUNT_IF(`dt_lancamento` = DATE'9999-12-31'),
    'dt_vencimento_duplicata', COUNT_IF(`dt_vencimento_duplicata` IS NULL), CAST(MIN(`dt_vencimento_duplicata`) AS STRING), CAST(MAX(`dt_vencimento_duplicata`) AS STRING), COUNT(DISTINCT `dt_vencimento_duplicata`), COUNT_IF(`dt_vencimento_duplicata` > current_date()), COUNT_IF(`dt_vencimento_duplicata` = DATE'9999-12-31'),
    'dt_pedido_compra', COUNT_IF(`dt_pedido_compra` IS NULL), CAST(MIN(`dt_pedido_compra`) AS STRING), CAST(MAX(`dt_pedido_compra`) AS STRING), COUNT(DISTINCT `dt_pedido_compra`), COUNT_IF(`dt_pedido_compra` > current_date()), COUNT_IF(`dt_pedido_compra` = DATE'9999-12-31'),
    'dt_criacao_remessa', COUNT_IF(`dt_criacao_remessa` IS NULL), CAST(MIN(`dt_criacao_remessa`) AS STRING), CAST(MAX(`dt_criacao_remessa`) AS STRING), COUNT(DISTINCT `dt_criacao_remessa`), COUNT_IF(`dt_criacao_remessa` > current_date()), COUNT_IF(`dt_criacao_remessa` = DATE'9999-12-31')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
)
ORDER BY coluna;

## 12.2 Campos de hora

In [0]:
-- CAMPOS DE HORA
SELECT coluna, vazios, fmt_HHMMSS, fmt_HH_MM, hora_zero, distintos
FROM (
  SELECT stack(2,
    'hr_hora', COUNT_IF(`hr_hora` IS NULL OR trim(`hr_hora`) = ''), COUNT_IF(trim(`hr_hora`) RLIKE '^[0-9]{6}$'), COUNT_IF(trim(`hr_hora`) RLIKE '^[0-9]{2}:[0-9]{2}'), COUNT_IF(trim(`hr_hora`) IN ('000000', '00:00:00')), COUNT(DISTINCT `hr_hora`),
    'hr_criacao_remessa', COUNT_IF(`hr_criacao_remessa` IS NULL OR trim(`hr_criacao_remessa`) = ''), COUNT_IF(trim(`hr_criacao_remessa`) RLIKE '^[0-9]{6}$'), COUNT_IF(trim(`hr_criacao_remessa`) RLIKE '^[0-9]{2}:[0-9]{2}'), COUNT_IF(trim(`hr_criacao_remessa`) IN ('000000', '00:00:00')), COUNT(DISTINCT `hr_criacao_remessa`)
  ) AS (coluna, vazios, fmt_HHMMSS, fmt_HH_MM, hora_zero, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(14,
    'nr_nota_fiscal_eletronica', 'string', COUNT_IF(CAST(`nr_nota_fiscal_eletronica` AS STRING) IS NULL OR trim(CAST(`nr_nota_fiscal_eletronica` AS STRING)) = ''), MIN(length(trim(CAST(`nr_nota_fiscal_eletronica` AS STRING)))), MAX(length(trim(CAST(`nr_nota_fiscal_eletronica` AS STRING)))), COUNT_IF(trim(CAST(`nr_nota_fiscal_eletronica` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`nr_nota_fiscal_eletronica` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`nr_nota_fiscal_eletronica` AS STRING)), '^0+', '')),
    'cd_serie', 'string', COUNT_IF(CAST(`cd_serie` AS STRING) IS NULL OR trim(CAST(`cd_serie` AS STRING)) = ''), MIN(length(trim(CAST(`cd_serie` AS STRING)))), MAX(length(trim(CAST(`cd_serie` AS STRING)))), COUNT_IF(trim(CAST(`cd_serie` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cd_serie` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cd_serie` AS STRING)), '^0+', '')),
    'cd_empresa', 'string', COUNT_IF(CAST(`cd_empresa` AS STRING) IS NULL OR trim(CAST(`cd_empresa` AS STRING)) = ''), MIN(length(trim(CAST(`cd_empresa` AS STRING)))), MAX(length(trim(CAST(`cd_empresa` AS STRING)))), COUNT_IF(trim(CAST(`cd_empresa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cd_empresa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cd_empresa` AS STRING)), '^0+', '')),
    'cd_fornecedor', 'string', COUNT_IF(CAST(`cd_fornecedor` AS STRING) IS NULL OR trim(CAST(`cd_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cd_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cd_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cd_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cd_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cd_fornecedor` AS STRING)), '^0+', '')),
    'cd_material', 'string', COUNT_IF(CAST(`cd_material` AS STRING) IS NULL OR trim(CAST(`cd_material` AS STRING)) = ''), MIN(length(trim(CAST(`cd_material` AS STRING)))), MAX(length(trim(CAST(`cd_material` AS STRING)))), COUNT_IF(trim(CAST(`cd_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cd_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cd_material` AS STRING)), '^0+', '')),
    'nr_documento_compras', 'string', COUNT_IF(CAST(`nr_documento_compras` AS STRING) IS NULL OR trim(CAST(`nr_documento_compras` AS STRING)) = ''), MIN(length(trim(CAST(`nr_documento_compras` AS STRING)))), MAX(length(trim(CAST(`nr_documento_compras` AS STRING)))), COUNT_IF(trim(CAST(`nr_documento_compras` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`nr_documento_compras` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`nr_documento_compras` AS STRING)), '^0+', '')),
    'nr_item_pedido', 'string', COUNT_IF(CAST(`nr_item_pedido` AS STRING) IS NULL OR trim(CAST(`nr_item_pedido` AS STRING)) = ''), MIN(length(trim(CAST(`nr_item_pedido` AS STRING)))), MAX(length(trim(CAST(`nr_item_pedido` AS STRING)))), COUNT_IF(trim(CAST(`nr_item_pedido` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`nr_item_pedido` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`nr_item_pedido` AS STRING)), '^0+', '')),
    'nr_item_nota_fiscal', 'string', COUNT_IF(CAST(`nr_item_nota_fiscal` AS STRING) IS NULL OR trim(CAST(`nr_item_nota_fiscal` AS STRING)) = ''), MIN(length(trim(CAST(`nr_item_nota_fiscal` AS STRING)))), MAX(length(trim(CAST(`nr_item_nota_fiscal` AS STRING)))), COUNT_IF(trim(CAST(`nr_item_nota_fiscal` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`nr_item_nota_fiscal` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`nr_item_nota_fiscal` AS STRING)), '^0+', '')),
    'nr_documento_material', 'string', COUNT_IF(CAST(`nr_documento_material` AS STRING) IS NULL OR trim(CAST(`nr_documento_material` AS STRING)) = ''), MIN(length(trim(CAST(`nr_documento_material` AS STRING)))), MAX(length(trim(CAST(`nr_documento_material` AS STRING)))), COUNT_IF(trim(CAST(`nr_documento_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`nr_documento_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`nr_documento_material` AS STRING)), '^0+', '')),
    'nr_remessa', 'string', COUNT_IF(CAST(`nr_remessa` AS STRING) IS NULL OR trim(CAST(`nr_remessa` AS STRING)) = ''), MIN(length(trim(CAST(`nr_remessa` AS STRING)))), MAX(length(trim(CAST(`nr_remessa` AS STRING)))), COUNT_IF(trim(CAST(`nr_remessa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`nr_remessa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`nr_remessa` AS STRING)), '^0+', '')),
    'cd_centro', 'string', COUNT_IF(CAST(`cd_centro` AS STRING) IS NULL OR trim(CAST(`cd_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cd_centro` AS STRING)))), MAX(length(trim(CAST(`cd_centro` AS STRING)))), COUNT_IF(trim(CAST(`cd_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cd_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cd_centro` AS STRING)), '^0+', '')),
    'cd_cnpj', 'string', COUNT_IF(CAST(`cd_cnpj` AS STRING) IS NULL OR trim(CAST(`cd_cnpj` AS STRING)) = ''), MIN(length(trim(CAST(`cd_cnpj` AS STRING)))), MAX(length(trim(CAST(`cd_cnpj` AS STRING)))), COUNT_IF(trim(CAST(`cd_cnpj` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cd_cnpj` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cd_cnpj` AS STRING)), '^0+', '')),
    'cd_chave_acesso_nfe', 'string', COUNT_IF(CAST(`cd_chave_acesso_nfe` AS STRING) IS NULL OR trim(CAST(`cd_chave_acesso_nfe` AS STRING)) = ''), MIN(length(trim(CAST(`cd_chave_acesso_nfe` AS STRING)))), MAX(length(trim(CAST(`cd_chave_acesso_nfe` AS STRING)))), COUNT_IF(trim(CAST(`cd_chave_acesso_nfe` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cd_chave_acesso_nfe` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cd_chave_acesso_nfe` AS STRING)), '^0+', '')),
    'nr_ordem', 'string', COUNT_IF(CAST(`nr_ordem` AS STRING) IS NULL OR trim(CAST(`nr_ordem` AS STRING)) = ''), MIN(length(trim(CAST(`nr_ordem` AS STRING)))), MAX(length(trim(CAST(`nr_ordem` AS STRING)))), COUNT_IF(trim(CAST(`nr_ordem` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`nr_ordem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`nr_ordem` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real,
não granularidade adicional.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`nr_nota_fiscal_eletronica` AS STRING), ''), COALESCE(CAST(`cd_serie` AS STRING), ''), COALESCE(CAST(`cd_empresa` AS STRING), ''), COALESCE(CAST(`cd_local_negocios` AS STRING), ''), COALESCE(CAST(`cd_fornecedor` AS STRING), ''), COALESCE(CAST(`cd_chave_acesso_nfe` AS STRING), ''), COALESCE(CAST(`nr_item_nota_fiscal` AS STRING), ''), COALESCE(CAST(`cd_material` AS STRING), ''), COALESCE(CAST(`cd_material_xml` AS STRING), ''), COALESCE(CAST(`nr_documento_compras` AS STRING), ''), COALESCE(CAST(`nr_item_pedido` AS STRING), ''), COALESCE(CAST(`nr_documento_faturamento` AS STRING), ''), COALESCE(CAST(`nr_exercicio` AS STRING), ''), COALESCE(CAST(`nr_documento_material` AS STRING), ''), COALESCE(CAST(`nr_remessa` AS STRING), ''), COALESCE(CAST(`cd_chave_acesso_dacte` AS STRING), ''), COALESCE(CAST(`cd_cnpj_transportador` AS STRING), ''), COALESCE(CAST(`cd_cnpj` AS STRING), ''), COALESCE(CAST(`cd_codigo_barras` AS STRING), ''), COALESCE(CAST(`qt_contagem_cte` AS STRING), ''), COALESCE(CAST(`nr_transporte` AS STRING), ''), COALESCE(CAST(`cd_centro` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`dt_recebimento` AS STRING), ''), COALESCE(CAST(`dt_recebimento_item` AS STRING), ''), COALESCE(CAST(`dt_justificativa` AS STRING), ''), COALESCE(CAST(`dt_data` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_vencimento_duplicata` AS STRING), ''), COALESCE(CAST(`hr_hora` AS STRING), ''), COALESCE(CAST(`cd_ncm` AS STRING), ''), COALESCE(CAST(`cd_cfop` AS STRING), ''), COALESCE(CAST(`cd_cst_icms` AS STRING), ''), COALESCE(CAST(`cd_iva` AS STRING), ''), COALESCE(CAST(`cd_origem_material` AS STRING), ''), COALESCE(CAST(`cd_procedencia_material` AS STRING), ''), COALESCE(CAST(`tx_descricao_produto` AS STRING), ''), COALESCE(CAST(`tx_descricao_material` AS STRING), ''), COALESCE(CAST(`qt_quantidade` AS STRING), ''), COALESCE(CAST(`qt_quantidade_xml` AS STRING), ''), COALESCE(CAST(`qt_quantidade_recebida` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_xml` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_recebimento` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_nfe` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario_bruto` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario_liquido` AS STRING), ''), COALESCE(CAST(`vl_unitario` AS STRING), ''), COALESCE(CAST(`vl_bruto` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`vl_total` AS STRING), ''), COALESCE(CAST(`pc_aliquota_icms` AS STRING), ''), COALESCE(CAST(`vl_base_icms` AS STRING), ''), COALESCE(CAST(`vl_icms` AS STRING), ''), COALESCE(CAST(`pc_aliquota_st` AS STRING), ''), COALESCE(CAST(`vl_base_st` AS STRING), ''), COALESCE(CAST(`vl_st` AS STRING), ''), COALESCE(CAST(`pc_aliquota_ipi` AS STRING), ''), COALESCE(CAST(`vl_base_ipi` AS STRING), ''), COALESCE(CAST(`vl_ipi` AS STRING), ''), COALESCE(CAST(`pc_aliquota_pis` AS STRING), ''), COALESCE(CAST(`vl_base_pis` AS STRING), ''), COALESCE(CAST(`vl_pis` AS STRING), ''), COALESCE(CAST(`pc_aliquota_cofins` AS STRING), ''), COALESCE(CAST(`vl_base_cofins` AS STRING), ''), COALESCE(CAST(`vl_cofins` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_icms_item` AS STRING), ''), COALESCE(CAST(`vl_total_bruto` AS STRING), ''), COALESCE(CAST(`vl_total_liquido` AS STRING), ''), COALESCE(CAST(`vl_total_icms` AS STRING), ''), COALESCE(CAST(`vl_total_st` AS STRING), ''), COALESCE(CAST(`vl_total_ipi` AS STRING), ''), COALESCE(CAST(`vl_total_produto` AS STRING), ''), COALESCE(CAST(`vl_despesas_acessorias` AS STRING), ''), COALESCE(CAST(`vl_frete` AS STRING), ''), COALESCE(CAST(`vl_icms_nao_destacado` AS STRING), ''), COALESCE(CAST(`vl_icms_st_nao_destacado` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_icms` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_st` AS STRING), ''), COALESCE(CAST(`qt_peso_total_nota_fiscal` AS STRING), ''), COALESCE(CAST(`qt_peso_liquido` AS STRING), ''), COALESCE(CAST(`cd_unidade_peso` AS STRING), ''), COALESCE(CAST(`qt_volumes` AS STRING), ''), COALESCE(CAST(`cd_incoterms` AS STRING), ''), COALESCE(CAST(`cd_modalidade_frete` AS STRING), ''), COALESCE(CAST(`nm_transportador` AS STRING), ''), COALESCE(CAST(`cd_frente` AS STRING), ''), COALESCE(CAST(`tx_status` AS STRING), ''), COALESCE(CAST(`st_simulacao` AS STRING), ''), COALESCE(CAST(`tx_log` AS STRING), ''), COALESCE(CAST(`tx_log_simulacao` AS STRING), ''), COALESCE(CAST(`tx_justificativa` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`cd_area_responsavel` AS STRING), ''), COALESCE(CAST(`tp_documento_compras` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`cd_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`dt_pedido_compra` AS STRING), ''), COALESCE(CAST(`vl_preco_liquido_pedido` AS STRING), ''), COALESCE(CAST(`nr_remessa_entrega` AS STRING), ''), COALESCE(CAST(`dt_criacao_remessa` AS STRING), ''), COALESCE(CAST(`hr_criacao_remessa` AS STRING), ''), COALESCE(CAST(`nr_ordem` AS STRING), ''), COALESCE(CAST(`cd_revisao_manutencao` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`nr_nota_fiscal_eletronica` AS STRING), ''), COALESCE(CAST(`cd_serie` AS STRING), ''), COALESCE(CAST(`cd_empresa` AS STRING), ''), COALESCE(CAST(`cd_local_negocios` AS STRING), ''), COALESCE(CAST(`cd_fornecedor` AS STRING), ''), COALESCE(CAST(`cd_chave_acesso_nfe` AS STRING), ''), COALESCE(CAST(`nr_item_nota_fiscal` AS STRING), ''), COALESCE(CAST(`cd_material` AS STRING), ''), COALESCE(CAST(`cd_material_xml` AS STRING), ''), COALESCE(CAST(`nr_documento_compras` AS STRING), ''), COALESCE(CAST(`nr_item_pedido` AS STRING), ''), COALESCE(CAST(`nr_documento_faturamento` AS STRING), ''), COALESCE(CAST(`nr_exercicio` AS STRING), ''), COALESCE(CAST(`nr_documento_material` AS STRING), ''), COALESCE(CAST(`nr_remessa` AS STRING), ''), COALESCE(CAST(`cd_chave_acesso_dacte` AS STRING), ''), COALESCE(CAST(`cd_cnpj_transportador` AS STRING), ''), COALESCE(CAST(`cd_cnpj` AS STRING), ''), COALESCE(CAST(`cd_codigo_barras` AS STRING), ''), COALESCE(CAST(`qt_contagem_cte` AS STRING), ''), COALESCE(CAST(`nr_transporte` AS STRING), ''), COALESCE(CAST(`cd_centro` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`dt_recebimento` AS STRING), ''), COALESCE(CAST(`dt_recebimento_item` AS STRING), ''), COALESCE(CAST(`dt_justificativa` AS STRING), ''), COALESCE(CAST(`dt_data` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_vencimento_duplicata` AS STRING), ''), COALESCE(CAST(`hr_hora` AS STRING), ''), COALESCE(CAST(`cd_ncm` AS STRING), ''), COALESCE(CAST(`cd_cfop` AS STRING), ''), COALESCE(CAST(`cd_cst_icms` AS STRING), ''), COALESCE(CAST(`cd_iva` AS STRING), ''), COALESCE(CAST(`cd_origem_material` AS STRING), ''), COALESCE(CAST(`cd_procedencia_material` AS STRING), ''), COALESCE(CAST(`tx_descricao_produto` AS STRING), ''), COALESCE(CAST(`tx_descricao_material` AS STRING), ''), COALESCE(CAST(`qt_quantidade` AS STRING), ''), COALESCE(CAST(`qt_quantidade_xml` AS STRING), ''), COALESCE(CAST(`qt_quantidade_recebida` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_xml` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_recebimento` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_nfe` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario_bruto` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario_liquido` AS STRING), ''), COALESCE(CAST(`vl_unitario` AS STRING), ''), COALESCE(CAST(`vl_bruto` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`vl_total` AS STRING), ''), COALESCE(CAST(`pc_aliquota_icms` AS STRING), ''), COALESCE(CAST(`vl_base_icms` AS STRING), ''), COALESCE(CAST(`vl_icms` AS STRING), ''), COALESCE(CAST(`pc_aliquota_st` AS STRING), ''), COALESCE(CAST(`vl_base_st` AS STRING), ''), COALESCE(CAST(`vl_st` AS STRING), ''), COALESCE(CAST(`pc_aliquota_ipi` AS STRING), ''), COALESCE(CAST(`vl_base_ipi` AS STRING), ''), COALESCE(CAST(`vl_ipi` AS STRING), ''), COALESCE(CAST(`pc_aliquota_pis` AS STRING), ''), COALESCE(CAST(`vl_base_pis` AS STRING), ''), COALESCE(CAST(`vl_pis` AS STRING), ''), COALESCE(CAST(`pc_aliquota_cofins` AS STRING), ''), COALESCE(CAST(`vl_base_cofins` AS STRING), ''), COALESCE(CAST(`vl_cofins` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_icms_item` AS STRING), ''), COALESCE(CAST(`vl_total_bruto` AS STRING), ''), COALESCE(CAST(`vl_total_liquido` AS STRING), ''), COALESCE(CAST(`vl_total_icms` AS STRING), ''), COALESCE(CAST(`vl_total_st` AS STRING), ''), COALESCE(CAST(`vl_total_ipi` AS STRING), ''), COALESCE(CAST(`vl_total_produto` AS STRING), ''), COALESCE(CAST(`vl_despesas_acessorias` AS STRING), ''), COALESCE(CAST(`vl_frete` AS STRING), ''), COALESCE(CAST(`vl_icms_nao_destacado` AS STRING), ''), COALESCE(CAST(`vl_icms_st_nao_destacado` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_icms` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_st` AS STRING), ''), COALESCE(CAST(`qt_peso_total_nota_fiscal` AS STRING), ''), COALESCE(CAST(`qt_peso_liquido` AS STRING), ''), COALESCE(CAST(`cd_unidade_peso` AS STRING), ''), COALESCE(CAST(`qt_volumes` AS STRING), ''), COALESCE(CAST(`cd_incoterms` AS STRING), ''), COALESCE(CAST(`cd_modalidade_frete` AS STRING), ''), COALESCE(CAST(`nm_transportador` AS STRING), ''), COALESCE(CAST(`cd_frente` AS STRING), ''), COALESCE(CAST(`tx_status` AS STRING), ''), COALESCE(CAST(`st_simulacao` AS STRING), ''), COALESCE(CAST(`tx_log` AS STRING), ''), COALESCE(CAST(`tx_log_simulacao` AS STRING), ''), COALESCE(CAST(`tx_justificativa` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`cd_area_responsavel` AS STRING), ''), COALESCE(CAST(`tp_documento_compras` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`cd_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`dt_pedido_compra` AS STRING), ''), COALESCE(CAST(`vl_preco_liquido_pedido` AS STRING), ''), COALESCE(CAST(`nr_remessa_entrega` AS STRING), ''), COALESCE(CAST(`dt_criacao_remessa` AS STRING), ''), COALESCE(CAST(`hr_criacao_remessa` AS STRING), ''), COALESCE(CAST(`nr_ordem` AS STRING), ''), COALESCE(CAST(`cd_revisao_manutencao` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`nr_nota_fiscal_eletronica` AS STRING), ''), COALESCE(CAST(`cd_serie` AS STRING), ''), COALESCE(CAST(`cd_empresa` AS STRING), ''), COALESCE(CAST(`cd_local_negocios` AS STRING), ''), COALESCE(CAST(`cd_fornecedor` AS STRING), ''), COALESCE(CAST(`cd_chave_acesso_nfe` AS STRING), ''), COALESCE(CAST(`nr_item_nota_fiscal` AS STRING), ''), COALESCE(CAST(`cd_material` AS STRING), ''), COALESCE(CAST(`cd_material_xml` AS STRING), ''), COALESCE(CAST(`nr_documento_compras` AS STRING), ''), COALESCE(CAST(`nr_item_pedido` AS STRING), ''), COALESCE(CAST(`nr_documento_faturamento` AS STRING), ''), COALESCE(CAST(`nr_exercicio` AS STRING), ''), COALESCE(CAST(`nr_documento_material` AS STRING), ''), COALESCE(CAST(`nr_remessa` AS STRING), ''), COALESCE(CAST(`cd_chave_acesso_dacte` AS STRING), ''), COALESCE(CAST(`cd_cnpj_transportador` AS STRING), ''), COALESCE(CAST(`cd_cnpj` AS STRING), ''), COALESCE(CAST(`cd_codigo_barras` AS STRING), ''), COALESCE(CAST(`qt_contagem_cte` AS STRING), ''), COALESCE(CAST(`nr_transporte` AS STRING), ''), COALESCE(CAST(`cd_centro` AS STRING), ''), COALESCE(CAST(`nm_centro` AS STRING), ''), COALESCE(CAST(`dt_documento` AS STRING), ''), COALESCE(CAST(`dt_recebimento` AS STRING), ''), COALESCE(CAST(`dt_recebimento_item` AS STRING), ''), COALESCE(CAST(`dt_justificativa` AS STRING), ''), COALESCE(CAST(`dt_data` AS STRING), ''), COALESCE(CAST(`dt_lancamento` AS STRING), ''), COALESCE(CAST(`dt_vencimento_duplicata` AS STRING), ''), COALESCE(CAST(`hr_hora` AS STRING), ''), COALESCE(CAST(`cd_ncm` AS STRING), ''), COALESCE(CAST(`cd_cfop` AS STRING), ''), COALESCE(CAST(`cd_cst_icms` AS STRING), ''), COALESCE(CAST(`cd_iva` AS STRING), ''), COALESCE(CAST(`cd_origem_material` AS STRING), ''), COALESCE(CAST(`cd_procedencia_material` AS STRING), ''), COALESCE(CAST(`tx_descricao_produto` AS STRING), ''), COALESCE(CAST(`tx_descricao_material` AS STRING), ''), COALESCE(CAST(`qt_quantidade` AS STRING), ''), COALESCE(CAST(`qt_quantidade_xml` AS STRING), ''), COALESCE(CAST(`qt_quantidade_recebida` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_xml` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_recebimento` AS STRING), ''), COALESCE(CAST(`cd_unidade_medida_nfe` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario_bruto` AS STRING), ''), COALESCE(CAST(`vl_preco_unitario_liquido` AS STRING), ''), COALESCE(CAST(`vl_unitario` AS STRING), ''), COALESCE(CAST(`vl_bruto` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`vl_total` AS STRING), ''), COALESCE(CAST(`pc_aliquota_icms` AS STRING), ''), COALESCE(CAST(`vl_base_icms` AS STRING), ''), COALESCE(CAST(`vl_icms` AS STRING), ''), COALESCE(CAST(`pc_aliquota_st` AS STRING), ''), COALESCE(CAST(`vl_base_st` AS STRING), ''), COALESCE(CAST(`vl_st` AS STRING), ''), COALESCE(CAST(`pc_aliquota_ipi` AS STRING), ''), COALESCE(CAST(`vl_base_ipi` AS STRING), ''), COALESCE(CAST(`vl_ipi` AS STRING), ''), COALESCE(CAST(`pc_aliquota_pis` AS STRING), ''), COALESCE(CAST(`vl_base_pis` AS STRING), ''), COALESCE(CAST(`vl_pis` AS STRING), ''), COALESCE(CAST(`pc_aliquota_cofins` AS STRING), ''), COALESCE(CAST(`vl_base_cofins` AS STRING), ''), COALESCE(CAST(`vl_cofins` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_icms_item` AS STRING), ''), COALESCE(CAST(`vl_total_bruto` AS STRING), ''), COALESCE(CAST(`vl_total_liquido` AS STRING), ''), COALESCE(CAST(`vl_total_icms` AS STRING), ''), COALESCE(CAST(`vl_total_st` AS STRING), ''), COALESCE(CAST(`vl_total_ipi` AS STRING), ''), COALESCE(CAST(`vl_total_produto` AS STRING), ''), COALESCE(CAST(`vl_despesas_acessorias` AS STRING), ''), COALESCE(CAST(`vl_frete` AS STRING), ''), COALESCE(CAST(`vl_icms_nao_destacado` AS STRING), ''), COALESCE(CAST(`vl_icms_st_nao_destacado` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_icms` AS STRING), ''), COALESCE(CAST(`vl_base_calculo_st` AS STRING), ''), COALESCE(CAST(`qt_peso_total_nota_fiscal` AS STRING), ''), COALESCE(CAST(`qt_peso_liquido` AS STRING), ''), COALESCE(CAST(`cd_unidade_peso` AS STRING), ''), COALESCE(CAST(`qt_volumes` AS STRING), ''), COALESCE(CAST(`cd_incoterms` AS STRING), ''), COALESCE(CAST(`cd_modalidade_frete` AS STRING), ''), COALESCE(CAST(`nm_transportador` AS STRING), ''), COALESCE(CAST(`cd_frente` AS STRING), ''), COALESCE(CAST(`tx_status` AS STRING), ''), COALESCE(CAST(`st_simulacao` AS STRING), ''), COALESCE(CAST(`tx_log` AS STRING), ''), COALESCE(CAST(`tx_log_simulacao` AS STRING), ''), COALESCE(CAST(`tx_justificativa` AS STRING), ''), COALESCE(CAST(`nm_usuario` AS STRING), ''), COALESCE(CAST(`cd_area_responsavel` AS STRING), ''), COALESCE(CAST(`tp_documento_compras` AS STRING), ''), COALESCE(CAST(`tp_classificacao_contabil` AS STRING), ''), COALESCE(CAST(`cd_grupo_mercadorias` AS STRING), ''), COALESCE(CAST(`dt_pedido_compra` AS STRING), ''), COALESCE(CAST(`vl_preco_liquido_pedido` AS STRING), ''), COALESCE(CAST(`nr_remessa_entrega` AS STRING), ''), COALESCE(CAST(`dt_criacao_remessa` AS STRING), ''), COALESCE(CAST(`hr_criacao_remessa` AS STRING), ''), COALESCE(CAST(`nr_ordem` AS STRING), ''), COALESCE(CAST(`cd_revisao_manutencao` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

## 15. Amostra de linhas completas

In [0]:
-- AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
ORDER BY `nr_nota_fiscal_eletronica`, `nr_item_nota_fiscal`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cd_centro
SELECT COALESCE(NULLIF(trim(CAST(`cd_centro` AS STRING)), ''), '(vazio)') AS `cd_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY COALESCE(NULLIF(trim(CAST(`cd_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cd_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cd_empresa` AS STRING)), ''), '(vazio)') AS `cd_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY COALESCE(NULLIF(trim(CAST(`cd_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cd_cfop
SELECT COALESCE(NULLIF(trim(CAST(`cd_cfop` AS STRING)), ''), '(vazio)') AS `cd_cfop`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY COALESCE(NULLIF(trim(CAST(`cd_cfop` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tx_status
SELECT COALESCE(NULLIF(trim(CAST(`tx_status` AS STRING)), ''), '(vazio)') AS `tx_status`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY COALESCE(NULLIF(trim(CAST(`tx_status` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cd_grupo_mercadorias
SELECT COALESCE(NULLIF(trim(CAST(`cd_grupo_mercadorias` AS STRING)), ''), '(vazio)') AS `cd_grupo_mercadorias`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY COALESCE(NULLIF(trim(CAST(`cd_grupo_mercadorias` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

⚠️ **Esta tabela não possui nenhuma coluna de auditoria de carga** — não há `dateingest`,
`dh_carga` nem equivalente.

**Consequência:** é impossível saber a que momento o dado se refere. Use o `DESCRIBE HISTORY`
da seção 1 para identificar a última gravação, e registre isso como limitação no relatório.

In [0]:
-- 17. FRESHNESS
-- Sem coluna de auditoria. Unica fonte e o historico Delta.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_log_zvcmm125 LIMIT 10;

## 18. Análises específicas — ZVCMM125

### 18.1 A PK declarada de 5 campos é realmente única?

Os comentários marcam como PK: `nr_nota_fiscal_eletronica + cd_serie + cd_empresa +
cd_fornecedor + cd_material`. O clustering só tem os dois primeiros.

In [0]:
-- 18.1 PK DECLARADA vs CLUSTERING
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
SELECT 'clustering (nota + empresa)' AS chave,
       (SELECT COUNT(*) FROM (SELECT DISTINCT nr_nota_fiscal_eletronica, cd_empresa FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)) AS distintos,
       t.total AS linhas
  FROM t
UNION ALL
SELECT 'PK declarada (5 campos)',
       (SELECT COUNT(*) FROM (SELECT DISTINCT nr_nota_fiscal_eletronica, cd_serie, cd_empresa,
                                              cd_fornecedor, cd_material FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)),
       t.total
  FROM t
UNION ALL
SELECT 'PK declarada + item da nota',
       (SELECT COUNT(*) FROM (SELECT DISTINCT nr_nota_fiscal_eletronica, cd_serie, cd_empresa,
                                              cd_fornecedor, cd_material, nr_item_nota_fiscal FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)),
       t.total
  FROM t;

In [0]:
-- 18.1b ITENS POR NOTA FISCAL
SELECT itens_na_nota, COUNT(*) AS notas
FROM (SELECT nr_nota_fiscal_eletronica, cd_serie, cd_empresa,
             COUNT(*) AS itens_na_nota
        FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
       GROUP BY nr_nota_fiscal_eletronica, cd_serie, cd_empresa)
GROUP BY itens_na_nota
ORDER BY itens_na_nota
LIMIT 40;

### 18.2 Quantidade no XML × recebida × do pedido

A tabela tem três quantidades distintas para a mesma linha. Divergência entre elas
é informação de negócio (recebimento parcial, divergência de nota), não erro.

In [0]:
-- 18.2 AS TRES QUANTIDADES
SELECT COUNT(*)                                                         AS linhas,
       COUNT_IF(qt_quantidade = qt_quantidade_xml)                      AS qtd_igual_xml,
       COUNT_IF(qt_quantidade <> qt_quantidade_xml)                     AS qtd_diferente_xml,
       COUNT_IF(qt_quantidade_recebida = qt_quantidade)                 AS recebida_igual_qtd,
       COUNT_IF(qt_quantidade_recebida < qt_quantidade)                 AS recebimento_parcial,
       COUNT_IF(qt_quantidade_recebida > qt_quantidade)                 AS recebimento_a_maior,
       COUNT_IF(qt_quantidade_recebida IS NULL OR qt_quantidade_recebida = 0) AS sem_recebimento
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

### 18.3 Coerência dos tributos

In [0]:
-- 18.3 COERENCIA DOS TRIBUTOS
SELECT 'ICMS' AS tributo,
       COUNT_IF(COALESCE(vl_base_icms,0) <> 0)                             AS com_base,
       COUNT_IF(COALESCE(vl_icms,0) <> 0)                                  AS com_valor,
       COUNT_IF(COALESCE(vl_base_icms,0) = 0 AND COALESCE(vl_icms,0) <> 0) AS valor_sem_base,
       COUNT_IF(ABS(COALESCE(vl_icms,0)
              - COALESCE(vl_base_icms,0)*COALESCE(pc_aliquota_icms,0)/100) > 0.05) AS formula_diverge
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
UNION ALL
SELECT 'ST', COUNT_IF(COALESCE(vl_base_st,0) <> 0), COUNT_IF(COALESCE(vl_st,0) <> 0),
       COUNT_IF(COALESCE(vl_base_st,0) = 0 AND COALESCE(vl_st,0) <> 0),
       COUNT_IF(ABS(COALESCE(vl_st,0) - COALESCE(vl_base_st,0)*COALESCE(pc_aliquota_st,0)/100) > 0.05)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
UNION ALL
SELECT 'IPI', COUNT_IF(COALESCE(vl_base_ipi,0) <> 0), COUNT_IF(COALESCE(vl_ipi,0) <> 0),
       COUNT_IF(COALESCE(vl_base_ipi,0) = 0 AND COALESCE(vl_ipi,0) <> 0),
       COUNT_IF(ABS(COALESCE(vl_ipi,0) - COALESCE(vl_base_ipi,0)*COALESCE(pc_aliquota_ipi,0)/100) > 0.05)
  FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

### 18.4 As quatro unidades de medida

A linha tem `cd_unidade_medida`, `_xml`, `_recebimento` e `_nfe`.
Divergência entre elas é esperada — é justamente o que a transação monitora.

In [0]:
-- 18.4 UNIDADES DE MEDIDA
SELECT COUNT(*)                                                                   AS linhas,
       COUNT_IF(COALESCE(cd_unidade_medida,'') = COALESCE(cd_unidade_medida_xml,''))  AS basica_igual_xml,
       COUNT_IF(COALESCE(cd_unidade_medida,'') <> COALESCE(cd_unidade_medida_xml,'')) AS basica_diferente_xml,
       COUNT_IF(COALESCE(cd_unidade_medida_recebimento,'')
              = COALESCE(cd_unidade_medida,''))                                    AS recebimento_igual_basica,
       COUNT_IF(trim(COALESCE(cd_unidade_medida_nfe,'')) = '')                     AS nfe_sem_unidade
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

### 18.5 Vínculos: pedido, remessa e ordem de manutenção

Esta tabela liga a NF-e ao pedido de compra, à remessa e à ordem de manutenção.
O campo `cd_revisao_manutencao` é o mesmo usado para distinguir GP de recorrente.

In [0]:
-- 18.5 VINCULOS
SELECT COUNT(*)                                                    AS linhas,
       COUNT_IF(trim(COALESCE(nr_documento_compras,'')) <> '')     AS com_pedido_compra,
       COUNT_IF(trim(COALESCE(nr_remessa,'')) <> '')               AS com_remessa,
       COUNT_IF(trim(COALESCE(nr_remessa_entrega,'')) <> '')       AS com_remessa_entrega,
       COUNT_IF(trim(COALESCE(nr_ordem,'')) <> '')                 AS com_ordem_manutencao,
       COUNT_IF(trim(COALESCE(cd_revisao_manutencao,'')) <> '')    AS com_revisao_manutencao,
       COUNT_IF(trim(COALESCE(nr_documento_material,'')) <> '')    AS com_documento_material
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125;

In [0]:
-- 18.5b REVISAO DE MANUTENCAO (GP vs recorrente)
SELECT COALESCE(NULLIF(trim(cd_revisao_manutencao), ''), '(vazio)') AS cd_revisao_manutencao,
       COUNT(*)                        AS linhas,
       COUNT(DISTINCT nr_ordem)        AS ordens,
       ROUND(SUM(vl_total), 2)         AS valor_total
FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
GROUP BY COALESCE(NULLIF(trim(cd_revisao_manutencao), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 30;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125
ORDER BY `nr_nota_fiscal_eletronica`, `nr_item_nota_fiscal`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125),
g AS (
  SELECT 'nr_nota_fiscal_eletronica' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
UNION ALL
  SELECT 'nr_nota_fiscal_eletronica + cd_serie + cd_empresa + cd_fornecedor' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica`, `cd_serie`, `cd_empresa`, `cd_fornecedor` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
UNION ALL
  SELECT 'nr_nota_fiscal_eletronica + cd_serie + cd_empresa + cd_fornecedor + cd_material' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica`, `cd_serie`, `cd_empresa`, `cd_fornecedor`, `cd_material` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
UNION ALL
  SELECT 'nr_nota_fiscal_eletronica + cd_serie + cd_empresa + cd_fornecedor + cd_material + nr_item_nota_fiscal' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `nr_nota_fiscal_eletronica`, `cd_serie`, `cd_empresa`, `cd_fornecedor`, `cd_material`, `nr_item_nota_fiscal` FROM dev_procurement.corp_curated.tbl_ds_log_zvcmm125)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'ZVCMM125' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'tbl_ds_log_zvcmm125'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '106'
UNION ALL SELECT 'IDENTIFICACAO', 'clustering declarado',
       'nr_nota_fiscal_eletronica, cd_empresa'
UNION ALL SELECT 'IDENTIFICACAO', 'coluna de recorte', 'cd_centro'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Gerar o notebook de cenário com esse recorte.
3. Extrair a ZVCMM125 no SAP com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] Alertas de partida verificados (seção 18)
